In [7]:
import os
import glob
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *
from spatial_manifolds.anaylsis_parameters import ngs_color, gc_color

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

fig_path = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_5_xgboost_distances/'
source_path = '/Users/harryclark/Downloads/COHORT12/'

cell_classifications = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# ── Analysis configuration ────────────────────────────────────────────────────
# Set to True/False to control which target cell types are computed.
# NGS targets are far more numerous and will take significantly longer to run.
CALCULATE_GC_TARGETS  = True   # predict grid cells (GC)
CALCULATE_NGS_TARGETS = True  # predict non-grid spatial cells (NGS)

# Set to True to skip all computation and only load rows from already-cached files.
# No new models will be fitted; sessions/cells with no cache file are silently skipped.
CACHE_ONLY = False
# ─────────────────────────────────────────────────────────────────────────────


## XGBoost shank assay — GC / NGS predicted by NGS cells per shank

### What this analysis does

For every session containing at least one grid cell (GC) and at least one non-grid spatial (NGS) cell, we fit XGBoost models to predict each **target cell's** (GC or NGS) firing rate in VR. Two covariate conditions are tested:

| `covariate_type` | Covariates |
|---|---|
| `pos` | Position alone (baseline) |
| `pos+ngs_shank` | Position + all NGS cells on a given shank |

Shank IDs (0–3) are assigned from `probe_x` via `reconstruct_shank_id`. Only shanks with at least one NGS cell in `tcs_time` contribute a `pos+ngs_shank` row.

---

### Key metric — ΔpR²

$$\Delta pR^2 = pR^2(\text{pos} + \text{cells}) - pR^2(\text{pos})$$

ΔpR² isolates the variance explained **by the NGS cells alone**, above and beyond what position accounts for. It is the primary quantity of interest throughout this notebook.

A positive ΔpR² means that knowing the activity of NGS cells on a particular shank improves prediction of the target cell's firing above and beyond position alone.

---

### Anchoring-mode split (second analysis)

Task-anchored labels are computed per target cell using spectral analysis of the trial-by-trial tuning curve:

1. Compute the spectrogram of the target cell's rate map via `spectral_analysis`.
2. Detect peak-frequency bins corresponding to task-locked harmonics (`[12, 28, 44, 60, 76]`).
3. Assign each trial a binary label (1 = anchored, 0 = non-anchored) via `get_kmeans_spatial_labels`.
4. Map trial labels to time bins and compute conditional pR² for each anchoring mode.

This lets us ask: **does ΔpR² differ depending on whether the target cell is in an anchored or non-anchored state?**

---

### Resulting dataframes

**`results_df`** — one row per model fit (full session):

| Column | Description |
|---|---|
| `mouse` / `day` | Session identifier |
| `target_cluster_id` | Cluster ID of the target cell |
| `target_shank_id` | Shank ID of the target cell (0–3) |
| `target_cell_type` | `GC` or `NG` |
| `cov_shank_id` | Shank of the NGS covariate pool (`NaN` for baseline) |
| `n_ngs_cells` | Number of NGS cells used as covariates |
| `covariate_type` | `pos` or `pos+ngs_shank` |
| `pR2_cv` | Cross-validated Poisson pseudo-R² |
| `mean_dist_to_ngs_um` | Mean 3-D distance (µm) from target to NGS covariates |

**`delta_df`** — one row per target × covariate shank (derived from `results_df`):

| Column | Description |
|---|---|
| `pR2_pos` | Baseline (position-only) pR² |
| `pR2_pos_ngs` | pR² with position + NGS on `cov_shank_id` |
| `delta_pR2` | ΔpR² = `pR2_pos_ngs` − `pR2_pos` |
| `same_shank` | Whether `cov_shank_id == target_shank_id` |

**`anchor_df`** — one row per target × anchoring mode (from separate anchoring computation):

| Column | Description |
|---|---|
| `anchor_mode` | `anchored` or `non_anchored` |
| `covariate_type` | `pos` or `pos+ngs_shank` (same-shank only) |
| `pR2_cond` | Conditional pR² evaluated on trials in that mode |
| `delta_pR2` | ΔpR² within that anchoring mode |
| `prop_anchor_mode` | Fraction of time bins in this mode |

In [9]:

# XGBoost model parameters
nfilters = 5
history_length = 1000  # ms

xgb_history = MLencoding(
    tunemodel='xgboost',
    cov_history=True,
    spike_history=False,
    window=time_bs,
    n_filters=nfilters,
    max_time=history_length,
)

all_rows = []

cache_dir = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost_cache'
os.makedirs(cache_dir, exist_ok=True)

# Sessions with at least 1 GC and at least 1 NGS cell
gc_sessions  = cell_classifications[cell_classifications['cell_type'] == 'GC'][['mouse', 'day']].drop_duplicates()
ngs_sessions = cell_classifications[cell_classifications['cell_type'] == 'NG'][['mouse', 'day']].drop_duplicates()
valid_sessions = gc_sessions.merge(ngs_sessions, on=['mouse', 'day'])
print(f"{len(valid_sessions)} sessions have at least 1 GC and 1 NGS cell.")

if CACHE_ONLY:
    # ── Cache-only mode: load every .csv that exists, skip everything else ────
    print("CACHE_ONLY=True — loading cached files only, no new models will be fitted.")
    _types_to_run = ['GC', 'NG']  # load all cached types regardless of flags
    for _, sess_row in valid_sessions.iterrows():
        mouse_id = int(sess_row['mouse'])
        day_id   = int(sess_row['day'])
        _sess_targets = cell_classifications[
            (cell_classifications['mouse'] == mouse_id) &
            (cell_classifications['day']   == day_id) &
            (cell_classifications['cell_type'].isin(_types_to_run))
        ]
        _loaded = 0
        for cid in _sess_targets['cluster_id']:
            cache_file = os.path.join(cache_dir, f'M{mouse_id}_D{day_id:02d}_C{int(cid)}.csv')
            if os.path.exists(cache_file):
                all_rows.extend(pd.read_csv(cache_file).to_dict('records'))
                _loaded += 1
        if _loaded:
            print(f"  M{mouse_id} D{day_id:02d}: loaded {_loaded} cached target(s).")
else:
    # ── Compute mode: fit models, using cache where available ─────────────────
    _types_to_run = (
        (['GC'] if CALCULATE_GC_TARGETS  else []) +
        (['NG'] if CALCULATE_NGS_TARGETS else [])
    )
    if not _types_to_run:
        raise ValueError("Both CALCULATE_GC_TARGETS and CALCULATE_NGS_TARGETS are False — nothing to run.")
    print(f"Running for target types: {_types_to_run}")

    expected_shanks = {0, 1, 2, 3}

    for _, sess_row in valid_sessions.iterrows():
        mouse_id = int(sess_row['mouse'])
        day_id   = int(sess_row['day'])
        print(f"\n=== M{mouse_id} D{day_id:02d} ===")

        # ── Session-level cache check: skip loading if all relevant targets are already cached ──
        _sess_targets_check = cell_classifications[
            (cell_classifications['mouse'] == mouse_id) &
            (cell_classifications['day']   == day_id) &
            (cell_classifications['cell_type'].isin(_types_to_run))
        ]
        _target_cache_files = {
            int(cid): os.path.join(cache_dir, f'M{mouse_id}_D{day_id:02d}_C{int(cid)}.csv')
            for cid in _sess_targets_check['cluster_id']
        }
        if len(_target_cache_files) > 0 and all(os.path.exists(f) for f in _target_cache_files.values()):
            for cid, cache_file in _target_cache_files.items():
                all_rows.extend(pd.read_csv(cache_file).to_dict('records'))
            print("  All targets cached — skipping session load.")
            continue

        # Load VR spike trains and behaviour
        tcs, tcs_time, _, last_ephys_bin, beh, clusters = compute_vr_tcs(
            mouse_id, day_id,
            apply_zscore=False, apply_guassian_filter=False,
            source_path=source_path,
        )

        last_ephys_time_bin = clusters[clusters.index[0]].count(
            bin_size=time_bs, time_units='ms'
        ).index[-1]
        ep = nap.IntervalSet(start=0, end=last_ephys_time_bin, time_units='s')

        dt_in_time  = np.array(
            beh['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=ep)
            - ((beh['trial_number'][0] - 1) * tl)
        )
        pos_in_time = dt_in_time % tl
        if np.any(np.isnan(pos_in_time)):
            pos_in_time = pd.Series(dt_in_time).ffill().bfill().values % tl

        # ── Session cell tables ────────────────────────────────────────────────────
        sess_gcs = cell_classifications[
            (cell_classifications['mouse'] == mouse_id) &
            (cell_classifications['day']   == day_id) &
            (cell_classifications['cell_type'] == 'GC')
        ].copy()
        sess_ngs = cell_classifications[
            (cell_classifications['mouse'] == mouse_id) &
            (cell_classifications['day']   == day_id) &
            (cell_classifications['cell_type'] == 'NG')
        ].copy()

        sess_gcs['cluster_id'] = sess_gcs['cluster_id'].astype(int)
        sess_ngs['cluster_id'] = sess_ngs['cluster_id'].astype(int)

        # ── Assign shank IDs from probe_x ─────────────────────────────────────────
        sess_gcs = reconstruct_shank_id(sess_gcs, mouse_id, colname='probe_x')
        sess_ngs = reconstruct_shank_id(sess_ngs, mouse_id, colname='probe_x')

        # Group NGS cells by shank — keep only shanks present in tcs_time
        ngs_by_shank = {
            shank: grp['cluster_id'].values.astype(int)
            for shank, grp in sess_ngs.groupby('shank_id')
            if any(c in tcs_time for c in grp['cluster_id'].astype(int))
        }

        # Require NGS cells on every shank
        if set(ngs_by_shank.keys()) != expected_shanks:
            print(f"  NGS cells not on all shanks (present: {sorted(ngs_by_shank.keys())}) — skipping.")
            continue

        print(f"  GCs: {len(sess_gcs)}  NGS: {len(sess_ngs)}  |  NGS shanks: { {s: len(v) for s,v in ngs_by_shank.items()} }")

        # ── Loop over target cells filtered by the config flags ───────────────────
        sess_targets = pd.concat([sess_gcs, sess_ngs], ignore_index=True)
        sess_targets = sess_targets[sess_targets['cell_type'].isin(_types_to_run)]

        for _, target_row in sess_targets.iterrows():
            target_id    = int(target_row['cluster_id'])
            target_shank = int(target_row['shank_id'])
            target_type  = str(target_row['cell_type'])

            # ── Target-level cache check ───────────────────────────────────────────
            cache_file = os.path.join(cache_dir, f'M{mouse_id}_D{day_id:02d}_C{target_id}.csv')
            if os.path.exists(cache_file):
                print(f"    {target_type} {target_id} (shank {target_shank}): loading from cache.")
                all_rows.extend(pd.read_csv(cache_file).to_dict('records'))
                continue

            if target_id not in tcs_time:
                continue

            y = np.array(tcs_time[target_id])
            T = len(y)
            pos = pos_in_time[:T]
            if len(pos) < T:
                pos = np.pad(pos, (0, T - len(pos)), mode='edge')

            tgt_x = float(target_row['SC_x'])
            tgt_y = float(target_row['SC_y'])
            tgt_z = float(target_row['SC_z'])

            target_rows = []

            # ── Baseline: position only ────────────────────────────────────────────
            _, pR2_cv = xgb_history.fit_cv(pos[:, None], y, verbose=0, continuous_folds=True)
            target_rows.append(dict(
                mouse=mouse_id, day=day_id,
                target_cluster_id=target_id,
                target_shank_id=target_shank,
                target_cell_type=target_type,
                cell_type=target_type,
                cov_shank_id=np.nan,
                n_ngs_cells=np.nan,
                covariate_type='pos',
                pR2_cv=float(np.nanmean(pR2_cv)),
                mean_dist_to_ngs_um=np.nan,
            ))

            # ── One model per NGS shank ────────────────────────────────────────────
            for cov_shank, ngs_cids in ngs_by_shank.items():

                # Keep only NGS cells present in tcs_time, excluding the target itself
                valid_ngs = [c for c in ngs_cids if c in tcs_time and c != target_id]
                if not valid_ngs:
                    continue

                # Stack NGS spike trains
                ngs_mat = np.vstack([
                    np.pad(
                        np.array(tcs_time[c])[:T],
                        (0, max(0, T - len(np.array(tcs_time[c])[:T]))),
                        mode='constant'
                    )
                    for c in valid_ngs
                ]).T  # shape (T, n_ngs)

                x = np.column_stack([pos, ngs_mat])

                # Mean Euclidean distance from target to each NGS cell on this shank
                ngs_coords = sess_ngs[sess_ngs['cluster_id'].isin(valid_ngs)][['SC_x', 'SC_y', 'SC_z']]
                dists = np.sqrt(
                    (ngs_coords['SC_x'].values.astype(float) - tgt_x)**2 +
                    (ngs_coords['SC_y'].values.astype(float) - tgt_y)**2 +
                    (ngs_coords['SC_z'].values.astype(float) - tgt_z)**2
                )
                mean_dist = float(np.nanmean(dists))

                _, pR2_cv = xgb_history.fit_cv(x, y, verbose=0, continuous_folds=True)
                target_rows.append(dict(
                    mouse=mouse_id, day=day_id,
                    target_cluster_id=target_id,
                    target_shank_id=target_shank,
                    target_cell_type=target_type,
                    cell_type=target_type,
                    cov_shank_id=int(cov_shank),
                    n_ngs_cells=len(valid_ngs),
                    covariate_type='pos+ngs_shank',
                    pR2_cv=float(np.nanmean(pR2_cv)),
                    mean_dist_to_ngs_um=mean_dist,
                ))

            # ── Save this target's rows to cache ──────────────────────────────────
            pd.DataFrame(target_rows).to_csv(cache_file, index=False)
            all_rows.extend(target_rows)
            print(f"    {target_type} {target_id} (shank {target_shank}): done.")

results_df = pd.DataFrame(all_rows)

# ── Backfill cell_type / target_cell_type from cell_classifications ────────
# Cache files saved before these columns were added won't have them.
_needs_backfill = (
    'cell_type' not in results_df.columns or results_df['cell_type'].isna().any() or
    'target_cell_type' not in results_df.columns or results_df['target_cell_type'].isna().any()
)
if _needs_backfill and len(results_df) > 0:
    _ct_lookup = (
        cell_classifications[['mouse', 'day', 'cluster_id', 'cell_type']]
        .rename(columns={'cluster_id': 'target_cluster_id'})
        .drop_duplicates(subset=['mouse', 'day', 'target_cluster_id'])
    )
    for _col in ['cell_type', 'target_cell_type']:
        if _col in results_df.columns:
            results_df = results_df.drop(columns=[_col])
    results_df = results_df.merge(_ct_lookup, on=['mouse', 'day', 'target_cluster_id'], how='left')
    results_df['target_cell_type'] = results_df['cell_type']
    print(f"Backfilled cell_type from cell_classifications: {results_df['cell_type'].value_counts().to_dict()}")

print(f"\nTotal rows collected: {len(results_df)}")
print(f"covariate_type counts:\n{results_df['covariate_type'].value_counts()}")
if 'cell_type' in results_df.columns:
    print(f"cell_type counts:\n{results_df['cell_type'].value_counts()}")
results_df.head(10)


66 sessions have at least 1 GC and 1 NGS cell.
Running for target types: ['GC', 'NG']

=== M20 D14 ===
  All targets cached — skipping session load.

=== M20 D15 ===
  All targets cached — skipping session load.

=== M20 D16 ===
  All targets cached — skipping session load.

=== M20 D17 ===
  NGS cells not on all shanks (present: [1, 2, 3]) — skipping.

=== M20 D18 ===
  All targets cached — skipping session load.

=== M20 D19 ===
  All targets cached — skipping session load.

=== M20 D20 ===
  NGS cells not on all shanks (present: [1]) — skipping.

=== M20 D21 ===
  NGS cells not on all shanks (present: [2]) — skipping.

=== M20 D23 ===
  NGS cells not on all shanks (present: [1]) — skipping.

=== M20 D24 ===
  NGS cells not on all shanks (present: [2]) — skipping.

=== M20 D25 ===
  NGS cells not on all shanks (present: [0]) — skipping.

=== M20 D26 ===
  NGS cells not on all shanks (present: [3]) — skipping.

=== M21 D15 ===
M21 unique probe_x values (before remap): [np.float64(250.

,mouse,day,target_cluster_id,target_shank_id,cov_shank_id,n_ngs_cells,covariate_type,pR2_cv,mean_dist_to_ngs_um,cell_type,target_cell_type
0,20,14,34,0,NaN,NaN,pos,0.027396,NaN,GC,GC
1,20,14,34,0,0.0,2.0,pos+ngs_shank,0.017168,105.000000,GC,GC
2,20,14,34,0,1.0,32.0,pos+ngs_shank,0.058554,272.320581,GC,GC
3,20,14,34,0,2.0,54.0,pos+ngs_shank,0.055879,410.645610,GC,GC
4,20,14,34,0,3.0,21.0,pos+ngs_shank,0.012833,548.142981,GC,GC
5,20,14,103,1,NaN,NaN,pos,0.016471,NaN,GC,GC
6,20,14,103,1,0.0,2.0,pos+ngs_shank,0.016918,393.745462,GC,GC
7,20,14,103,1,1.0,32.0,pos+ngs_shank,0.069928,207.187500,GC,GC
8,20,14,103,1,2.0,54.0,pos+ngs_shank,0.065179,326.102353,GC,GC
9,20,14,103,1,3.0,21.0,pos+ngs_shank,0.022341,401.951774,GC,GC


In [10]:
save_path = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost.csv'
results_df.to_csv(save_path, index=False)
print(f"Saved {len(results_df)} rows → {save_path}")


Saved 10028 rows → /Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost.csv


In [11]:

# ── Derive delta_pR2 = pR2(pos+ngs_shank) − pR2(pos) ─────────────────────
# Join the two covariate conditions on target identity + covariate shank.
# This is the primary metric that isolates the NGS cell contribution.

_pos_rows = (
    results_df[results_df['covariate_type'] == 'pos']
    [['mouse', 'day', 'target_cluster_id', 'target_shank_id', 'target_cell_type', 'pR2_cv']]
    .rename(columns={'pR2_cv': 'pR2_pos'})
)

_ngs_rows = (
    results_df[results_df['covariate_type'] == 'pos+ngs_shank']
    [['mouse', 'day', 'target_cluster_id', 'target_shank_id', 'target_cell_type',
      'cov_shank_id', 'n_ngs_cells', 'pR2_cv', 'mean_dist_to_ngs_um']]
    .rename(columns={'pR2_cv': 'pR2_pos_ngs'})
)

delta_df = _ngs_rows.merge(
    _pos_rows,
    on=['mouse', 'day', 'target_cluster_id', 'target_shank_id', 'target_cell_type'],
    how='inner',
)
delta_df['delta_pR2'] = delta_df['pR2_pos_ngs'] - delta_df['pR2_pos']
delta_df['same_shank'] = (delta_df['cov_shank_id'] == delta_df['target_shank_id'])

print(f"delta_df: {len(delta_df)} rows")
for ttype in ['GC', 'NG']:
    same = delta_df[(delta_df['same_shank']) & (delta_df['target_cell_type'] == ttype)]['delta_pR2']
    print(f"  {ttype} same-shank  ΔpR² — mean: {same.mean():.4f}  median: {same.median():.4f}  n={len(same)}")
delta_df.head()


delta_df: 8022 rows
  GC same-shank  ΔpR² — mean: 0.0737  median: 0.0412  n=246
  NG same-shank  ΔpR² — mean: 0.0555  median: 0.0285  n=1758


,mouse,day,target_cluster_id,target_shank_id,target_cell_type,cov_shank_id,n_ngs_cells,pR2_pos_ngs,mean_dist_to_ngs_um,pR2_pos,delta_pR2,same_shank
0,20,14,34,0,GC,0.0,2.0,0.017168,105.000000,0.027396,-0.010228,True
1,20,14,34,0,GC,1.0,32.0,0.058554,272.320581,0.027396,0.031158,False
2,20,14,34,0,GC,2.0,54.0,0.055879,410.645610,0.027396,0.028483,False
3,20,14,34,0,GC,3.0,21.0,0.012833,548.142981,0.027396,-0.014563,False
4,20,14,103,1,GC,0.0,2.0,0.016918,393.745462,0.016471,0.000447,False


## Anchoring-mode split — ΔpR² for anchored vs non-anchored trials

For each target cell (GC and NGS), task-anchored labels are computed from its own trial-by-trial tuning curve using spectral analysis (peak-frequency detection + k-means spatial labelling, identical to the approach in the task-anchoring figures). Each trial is labelled **anchored (1)** or **non-anchored (0)**.

For the anchoring-mode analysis, XGBoost models are fit for each target cell using:
- `pos` (position only, baseline)
- `pos+ngs_shank` (position plus all NGS cells from **every shank with at least one NGS cell**, not just the same shank)

Predictions are then evaluated **separately** on anchored and non-anchored time bins, giving conditional pR² values:

$$pR^2_{\text{TA}} = pR^2 \text{ evaluated on anchored trials}$$
$$pR^2_{\text{NTA}} = pR^2 \text{ evaluated on non-anchored trials}$$

The conditional ΔpR² within each mode is:

$$\Delta pR^2_{\text{mode}} = pR^2_{\text{pos+cells,\,mode}} - pR^2_{\text{pos,\,mode}}$$

Cells or modes with fewer than `MIN_ANCHOR_TRIALS` time bins are set to NaN.

Results are cached per target cell in `gc_ngs_shank_xgboost_anchor_cache/`, with one row per (target cell, covariate shank, anchoring mode).

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# Anchoring-mode conditional ΔpR² computation (all shanks)
# ══════════════════════════════════════════════════════════════════════════════

# ── Config ────────────────────────────────────────────────────────────────────
MIN_ANCHOR_TRIALS = 10      # minimum time-bins per anchoring mode
ANCHOR_CACHE_ONLY = False   # set True to load existing cache files only
ANCHOR_GC_TARGETS  = True
ANCHOR_NGS_TARGETS = True

anchor_cache_dir = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost_anchor_cache'
os.makedirs(anchor_cache_dir, exist_ok=True)

_anchor_types = (
    (['GC'] if ANCHOR_GC_TARGETS  else []) +
    (['NG'] if ANCHOR_NGS_TARGETS else [])
)

# ── Spectral-analysis peak indices (task harmonics) ───────────────────────────
PEAK_INDICES = [12, 28, 44, 60, 76]

def _compute_anchored_labels(target_id, tcs, last_ephys_bin):
    """Return per-trial task-anchored labels (1=anchored, 0=non-anchored)
    computed from the target cell's own trial-by-trial tuning curve."""
    if target_id not in tcs:
        return None
    tc = gaussian_filter(np.nan_to_num(tcs[target_id]).astype(np.float64), sigma=2.5)
    tc = tc[:last_ephys_bin]
    tcs_to_use = {target_id: tc, 1000: tc}
    results      = spectral_analysis(tcs_to_use, tl, bs=bs)
    S            = results[3].mean(0)
    max_peaks    = np.argmax(S, axis=0)
    labels_idx   = np.isin(max_peaks, PEAK_INDICES).astype(int)
    return get_kmeans_spatial_labels(tc, labels_idx, bs=bs, tl=tl)

# ── Main loop ─────────────────────────────────────────────────────────────────
anchor_rows = []

for _, sess_row in valid_sessions.iterrows():
    mouse_id = int(sess_row['mouse'])
    day_id   = int(sess_row['day'])

    sess_cells = cell_classifications[
        (cell_classifications['mouse']     == mouse_id) &
        (cell_classifications['day']       == day_id) &
        (cell_classifications['cell_type'].isin(_anchor_types))
    ].copy()
    if len(sess_cells) == 0:
        continue

    sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
    sess_cells = reconstruct_shank_id(sess_cells, mouse_id, colname='probe_x')

    cache_files = {
        int(r['cluster_id']): os.path.join(
            anchor_cache_dir,
            f'M{mouse_id}_D{day_id:02d}_C{int(r["cluster_id"])}_anchor.csv'
        )
        for _, r in sess_cells.iterrows()
    }

    # ── Session-level cache shortcut ──────────────────────────────────────────
    if all(os.path.exists(f) for f in cache_files.values()):
        for cf in cache_files.values():
            anchor_rows.extend(pd.read_csv(cf).to_dict('records'))
        print(f"M{mouse_id} D{day_id:02d}: all anchor targets cached — skipping.")
        continue

    if ANCHOR_CACHE_ONLY:
        for cf in cache_files.values():
            if os.path.exists(cf):
                anchor_rows.extend(pd.read_csv(cf).to_dict('records'))
        continue

    print(f"\n=== Anchor M{mouse_id} D{day_id:02d} ===")

    # ── Load session ──────────────────────────────────────────────────────────
    try:
        tcs, tcs_time, _, last_ephys_bin, beh, clusters = compute_vr_tcs(
            mouse_id, day_id,
            apply_zscore=False, apply_guassian_filter=False,
            source_path=source_path,
        )
    except Exception as e:
        print(f"  Load failed: {e}")
        continue

    last_ephys_time_bin = clusters[clusters.index[0]].count(
        bin_size=time_bs, time_units='ms'
    ).index[-1]
    ep         = nap.IntervalSet(start=0, end=last_ephys_time_bin, time_units='s')
    dt_in_time = np.array(
        beh['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=ep)
        - ((beh['trial_number'][0] - 1) * tl)
    )
    pos_in_time_anchor = dt_in_time % tl
    if np.any(np.isnan(pos_in_time_anchor)):
        pos_in_time_anchor = pd.Series(dt_in_time).ffill().bfill().values % tl

    trial_number_in_time = (dt_in_time // tl) + beh['trial_number'][0]
    first_trial = int(beh['trial_number'][0])

    # NGS by shank
    sess_ngs = cell_classifications[
        (cell_classifications['mouse']     == mouse_id) &
        (cell_classifications['day']       == day_id) &
        (cell_classifications['cell_type'] == 'NG')
    ].copy()
    sess_ngs['cluster_id'] = sess_ngs['cluster_id'].astype(int)
    sess_ngs = reconstruct_shank_id(sess_ngs, mouse_id, colname='probe_x')
    ngs_by_shank = {
        sh: grp['cluster_id'].values.astype(int)
        for sh, grp in sess_ngs.groupby('shank_id')
        if any(c in tcs_time for c in grp['cluster_id'].astype(int))
    }

    # ── Per-target loop ───────────────────────────────────────────────────────
    for _, target_row in sess_cells.iterrows():
        target_id    = int(target_row['cluster_id'])
        target_shank = int(target_row['shank_id'])
        target_type  = str(target_row['cell_type'])

        cache_file = cache_files[target_id]
        if os.path.exists(cache_file):
            anchor_rows.extend(pd.read_csv(cache_file).to_dict('records'))
            continue

        if target_id not in tcs_time:
            continue

        # ── Anchored labels ───────────────────────────────────────────────────
        try:
            task_anchored_labels = _compute_anchored_labels(target_id, tcs, last_ephys_bin)
        except Exception as e:
            print(f"  Anchoring failed for {target_id}: {e}")
            continue
        if task_anchored_labels is None:
            continue

        # Map trial labels → time bins
        n_label_trials = len(task_anchored_labels)
        tl_in_time = np.zeros(len(trial_number_in_time), dtype=int)
        for ti, tn in enumerate(trial_number_in_time.astype(int)):
            idx = tn - first_trial
            if 0 <= idx < n_label_trials:
                tl_in_time[ti] = int(task_anchored_labels[idx])

        y   = np.array(tcs_time[target_id])
        T   = len(y)
        pos = pos_in_time_anchor[:T]
        if len(pos) < T:
            pos = np.pad(pos, (0, T - len(pos)), mode='edge')
        tl_t = tl_in_time[:T]

        # ── Fit pos-only model ────────────────────────────────────────────────
        Y_hat_pos, _ = xgb_history.fit_cv(pos[:, None], y, verbose=0, continuous_folds=True)

        # ── For every available shank with NGS cells ─────────────────────────
        for cov_shank, ngs_cids in ngs_by_shank.items():
            valid_ngs = [c for c in ngs_cids if c in tcs_time and c != target_id]
            if not valid_ngs:
                continue
            ngs_mat = np.vstack([
                np.pad(
                    np.array(tcs_time[c])[:T],
                    (0, max(0, T - len(np.array(tcs_time[c])[:T]))),
                    mode='constant',
                )
                for c in valid_ngs
            ]).T
            x_ngs = np.column_stack([pos, ngs_mat])
            Y_hat_ngs, _ = xgb_history.fit_cv(x_ngs, y, verbose=0, continuous_folds=True)

            # ── Compute conditional pR² per anchoring mode ─────────────────--
            for anchor_name, label_val in [('anchored', 1), ('non_anchored', 0)]:
                mask = tl_t == label_val
                prop = float(mask.sum()) / T

                # pos baseline
                if mask.sum() >= MIN_ANCHOR_TRIALS:
                    pR2_pos_cond = float(poisson_pseudoR2(
                        y[mask], Y_hat_pos[mask], np.nanmean(y[mask])
                    ))
                    pR2_ngs_cond = float(poisson_pseudoR2(
                        y[mask], Y_hat_ngs[mask], np.nanmean(y[mask])
                    ))
                else:
                    pR2_pos_cond = np.nan
                    pR2_ngs_cond = np.nan

                delta = (
                    pR2_ngs_cond - pR2_pos_cond
                    if not (np.isnan(pR2_ngs_cond) or np.isnan(pR2_pos_cond))
                    else np.nan
                )
                anchor_rows.append(dict(
                    mouse=mouse_id, day=day_id,
                    target_cluster_id=target_id,
                    target_shank_id=target_shank,
                    target_cell_type=target_type,
                    cov_shank_id=int(cov_shank),
                    n_ngs_cells=len(valid_ngs),
                    covariate_type='pos+ngs_shank',
                    anchor_mode=anchor_name,
                    prop_anchor_mode=prop,
                    pR2_cond=pR2_ngs_cond,
                    delta_pR2=delta,
                ))

        # Also store pos-only baseline for each anchor mode (cov_shank_id=NaN)
        for anchor_name, label_val in [('anchored', 1), ('non_anchored', 0)]:
            mask = tl_t == label_val
            prop = float(mask.sum()) / T
            if mask.sum() >= MIN_ANCHOR_TRIALS:
                pR2_pos_cond = float(poisson_pseudoR2(
                    y[mask], Y_hat_pos[mask], np.nanmean(y[mask])
                ))
            else:
                pR2_pos_cond = np.nan
            anchor_rows.append(dict(
                mouse=mouse_id, day=day_id,
                target_cluster_id=target_id,
                target_shank_id=target_shank,
                target_cell_type=target_type,
                cov_shank_id=np.nan,
                n_ngs_cells=np.nan,
                covariate_type='pos',
                anchor_mode=anchor_name,
                prop_anchor_mode=prop,
                pR2_cond=pR2_pos_cond,
                delta_pR2=np.nan,
            ))

        pd.DataFrame([r for r in anchor_rows if r['target_cluster_id'] == target_id]).to_csv(cache_file, index=False)
        print(f"  {target_type} {target_id} (sh{target_shank}): done")

anchor_df = pd.DataFrame(anchor_rows)
print(f"\nanchor_df: {len(anchor_df)} rows")
print(anchor_df['anchor_mode'].value_counts())
print(anchor_df['cov_shank_id'].value_counts())
anchor_df.head(10)



=== Anchor M20 D14 ===
  GC 34 (sh0): done
  GC 103 (sh1): done
  GC 400 (sh3): done
  GC 412 (sh3): done
  NG 37 (sh0): done


In [ ]:

import seaborn as sns
from scipy import stats as _scipy_stats

# ══════════════════════════════════════════════════════════════════════════════
# Visualise ΔpR² for anchored vs non-anchored modes
# Rows = GC targets / NGS targets
# Columns = (a) box/strip by anchor mode  (b) paired scatter  (c) delta difference
# ══════════════════════════════════════════════════════════════════════════════

_target_types  = ['GC', 'NG']
_type_colors   = {'GC': gc_color, 'NG': ngs_color}
_type_labels   = {'GC': 'Grid cell (GC)', 'NG': 'Non-grid spatial (NGS)'}
_mode_colors   = {'anchored': '#9d5391', 'non_anchored': '#e7d7e8'}
_mode_labels   = {'anchored': 'Anchored', 'non_anchored': 'Non-anchored'}
_mode_order    = ['anchored', 'non_anchored']

# Pull out same-shank ΔpR² rows from anchor_df
_adelta = anchor_df[
    (anchor_df['covariate_type'] == 'pos+ngs_shank') &
    anchor_df['delta_pR2'].notna()
].copy()

fig, axes = plt.subplots(
    len(_target_types), 3,
    figsize=(12, 4.5 * len(_target_types)),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.35},
)
fig.suptitle('ΔpR² = pR²(pos+cells) − pR²(pos)  |  same-shank NGS covariates',
             fontsize=12, fontweight='bold', y=1.01)

for row_i, ttype in enumerate(_target_types):
    _col  = _type_colors[ttype]
    _lbl  = _type_labels[ttype]
    _sub  = _adelta[_adelta['target_cell_type'] == ttype]

    # ── (a) Box + strip: ΔpR² by anchoring mode ──────────────────────────────
    ax = axes[row_i, 0]
    sns.boxplot(
        data=_sub, x='anchor_mode', y='delta_pR2',
        order=_mode_order,
        palette=_mode_colors,
        width=0.55, fliersize=0, linecolor='#333333', ax=ax,
    )
    sns.stripplot(
        data=_sub, x='anchor_mode', y='delta_pR2',
        order=_mode_order,
        palette={'anchored': '#9d5391', 'non_anchored': '#b09ab5'},
        alpha=0.25, jitter=True, size=2.5, ax=ax, legend=False,
    )
    ax.axhline(0, color='#888', lw=1.0, ls='--')
    ax.set_xticks([0, 1])
    ax.set_xticklabels([_mode_labels[m] for m in _mode_order], fontsize=9)
    ax.set_xlabel('Anchoring mode', fontsize=9)
    ax.set_ylabel('ΔpR²', fontsize=10)
    ax.set_title(f'{_lbl}', fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Annotate N (sessions) and n (cells)
    _N = (_sub['mouse'].astype(str) + '_' + _sub['day'].astype(str)).nunique()
    _n = _sub['target_cluster_id'].nunique()
    ax.text(0.98, 0.98, f'N={_N} sessions\nn={_n} cells',
            ha='right', va='top', fontsize=8, color='grey',
            transform=ax.transAxes)

    # Wilcoxon signed-rank test (anchored vs non-anchored, paired per cell)
    _paired = _sub.pivot_table(
        index=['mouse', 'day', 'target_cluster_id'],
        columns='anchor_mode', values='delta_pR2',
    ).dropna()
    if len(_paired) >= 5:
        _wstat, _wpval = _scipy_stats.wilcoxon(_paired['anchored'], _paired['non_anchored'])
        _pstr = f'p={_wpval:.3g}' if _wpval >= 0.001 else 'p<0.001'
        ax.text(0.5, 0.03, f'Wilcoxon {_pstr}  (n={len(_paired)} paired)',
                ha='center', va='bottom', fontsize=7.5, color='#555',
                transform=ax.transAxes)

    # ── (b) Paired scatter: anchored ΔpR² vs non-anchored ΔpR² per cell ──────
    ax = axes[row_i, 1]
    if len(_paired) > 0:
        _anch  = _paired['anchored'].values
        _nanch = _paired['non_anchored'].values
        ax.scatter(_nanch, _anch, color=_col, alpha=0.35, s=12, linewidths=0)
        _lo = min(_anch.min(), _nanch.min()) - 0.02
        _hi = max(_anch.max(), _nanch.max()) + 0.02
        ax.plot([_lo, _hi], [_lo, _hi], color='#888', lw=1.0, ls='--')
        # regression line
        if len(_paired) >= 5:
            _slope, _intercept, _r, _p, _ = _scipy_stats.linregress(_nanch, _anch)
            _xs = np.array([_lo, _hi])
            ax.plot(_xs, _intercept + _slope * _xs, color=_col, lw=1.5)
            ax.text(0.97, 0.05,
                    f'r={_r:.2f}, p={_p:.2g}',
                    ha='right', va='bottom', fontsize=7.5, color=_col,
                    transform=ax.transAxes)
    ax.set_xlabel('ΔpR²  (non-anchored)', fontsize=9)
    ax.set_ylabel('ΔpR²  (anchored)',      fontsize=9)
    ax.set_title('Paired per cell', fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # ── (c) Difference: Δ(ΔpR²) = anchored − non-anchored ───────────────────
    ax = axes[row_i, 2]
    if len(_paired) > 0:
        _diff = _paired['anchored'].values - _paired['non_anchored'].values
        sns.histplot(_diff, bins=25, color=_col, alpha=0.65, ax=ax, edgecolor='white')
        ax.axvline(0,            color='#888', lw=1.0, ls='--')
        ax.axvline(_diff.mean(), color=_col,  lw=1.5, ls='-',
                   label=f'mean={_diff.mean():.3f}')
        ax.set_xlabel('Δ(ΔpR²)  [anchored − non-anchored]', fontsize=9)
        ax.set_ylabel('Cell count', fontsize=9)
        ax.set_title('ΔpR² difference distribution', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8, frameon=False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(
        '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_8_anchored_xgboost/',
        'delta_pR2_anchored_vs_nonanchored.pdf',
    ),
    dpi=300, bbox_inches='tight',
)
plt.show()


In [ ]:

import statsmodels.formula.api as smf

# ── Prepare identifiers for the nested hierarchy ───────────────────────────
lmm_df = results_df.copy()
lmm_df['mouse_day'] = (
    lmm_df['mouse'].astype(str) + '_' + lmm_df['day'].astype(str)
)
lmm_df['mouse_day_cluster'] = (
    lmm_df['mouse_day'] + '_' + lmm_df['target_cluster_id'].astype(str)
)

# Use float (not Int64) so NaN is preserved and numpy/statsmodels can handle it
lmm_df['cov_shank_id'] = lmm_df['cov_shank_id'].astype(float)

# ── Fit a LMM for a given covariate_type subset ───────────────────────────
def fit_lmm(df_sub, formula, groups_col='mouse', label=''):
    """Fit a Mixed Linear Model and return the result object.

    Hierarchy:
      Level 1 – mouse (groups), intercept-only random effect (re_formula='1')
      Level 2 – session (mouse_day) via vc_formula
      Level 3 – cell    (mouse_day_cluster) via vc_formula
    """
    md = smf.mixedlm(
        formula,
        data=df_sub,
        groups=df_sub[groups_col],
        re_formula="1",
        vc_formula={
            "mouse_day":         "0 + C(mouse_day)",
            "mouse_day_cluster": "0 + C(mouse_day_cluster)",
        },
    )
    result = md.fit(method='lbfgs', maxiter=1000)
    print(f"\n{'='*60}")
    print(f"  LMM: {label}")
    print(f"  Formula : {formula}")
    print(f"  N obs   : {len(df_sub)}")
    print(f"{'='*60}")
    print(result.summary())
    return result


lmm_results = {}

# ── pos+ngs_shank: fixed effects = shank identity + pool size ─────────────
df_cov = lmm_df[lmm_df['covariate_type'] == 'pos+ngs_shank'].dropna(
    subset=['cov_shank_id', 'n_ngs_cells', 'pR2_cv']
).copy()

lmm_results['pos+ngs_shank'] = fit_lmm(
    df_cov,
    formula='pR2_cv ~ C(cov_shank_id) + n_ngs_cells',
    label='pos+ngs_shank',
)

# ── pos (baseline): intercept-only — characterises GC-level variability ───
df_pos = lmm_df[lmm_df['covariate_type'] == 'pos'].dropna(
    subset=['pR2_cv']
).copy()

lmm_results['pos'] = fit_lmm(
    df_pos,
    formula='pR2_cv ~ 1',
    label='pos (baseline, intercept-only)',
)

# ── Save fixed-effects tables ──────────────────────────────────────────────
# Build from result attributes directly — MixedLMResults has no summary2()
fe_rows = []
for cov_type, res in lmm_results.items():
    fe_df = pd.DataFrame({
        'coef':    res.fe_params,
        'se':      res.bse_fe,
        'z':       res.tvalues,
        'p':       res.pvalues,
        'ci_low':  res.conf_int().iloc[:len(res.fe_params), 0],
        'ci_high': res.conf_int().iloc[:len(res.fe_params), 1],
    })
    fe_df.index.name = 'term'
    fe_df = fe_df.reset_index()
    fe_df.insert(0, 'covariate_type', cov_type)
    fe_rows.append(fe_df)

fe_table = pd.concat(fe_rows, ignore_index=True)
lmm_save_path = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_lmm_fixed_effects.csv'
fe_table.to_csv(lmm_save_path, index=False)
print(f"\nFixed-effects table saved → {lmm_save_path}")
fe_table


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy.spatial import cKDTree
from scipy import stats

# --- Panel and bin setup ---
shank_order = [1, 2, 3, 4]
bin_start  = 2800
bin_max    = 4000
bin_step   = 200
bin_edges  = np.arange(bin_start, bin_max, bin_step)
bin_labels = [f'{int(bin_edges[i])}–{int(bin_edges[i+1])}' for i in range(len(bin_edges) - 1)]

target_types = ['GC', 'NG']
_type_color  = {'GC': gc_color,  'NG': ngs_color}
_type_label  = {'GC': 'GC',     'NG': 'NGS'}

_figsize          = (16, 7)
_box_width_hue    = 1.0
_box_width_single = 1.0

fig, axes = plt.subplots(2, 5, figsize=_figsize, 
                         gridspec_kw={'hspace': 0.18, 
                                      'wspace': 0.3,
                                      'width_ratios': [1, 0.5, 1, 0.5, 1]}, sharey=False)
fig.subplots_adjust(left=0.13, right=0.98, top=0.92, bottom=0.13)

for col, _ttype in enumerate(target_types):
    _col    = _type_color[_ttype]
    _lbl    = _type_label[_ttype]
    _hue_order = ['pos', 'pos+NGS']
    _palette   = {'pos': 'white', 'pos+NGS': _col}
    _dodge     = _box_width_hue / 4

    # Prep: pR² — pos vs same-shank pos+NGS, by target shank
    _bl = (
        results_df[
            (results_df['covariate_type'] == 'pos') &
            (results_df['target_cell_type'] == _ttype)
        ][['mouse', 'day', 'target_cluster_id', 'pR2_cv', 'target_shank_id']].copy()
    )
    _bl['target_shank_id'] = _bl['target_shank_id'] + 1
    _bl['condition'] = 'pos'
    _ngs = (
        results_df[
            (results_df['covariate_type'] == 'pos+ngs_shank') &
            (results_df['cov_shank_id'] == results_df['target_shank_id']) &
            (results_df['target_cell_type'] == _ttype)
        ][['mouse', 'day', 'target_cluster_id', 'pR2_cv', 'target_shank_id']].copy()
    )
    _ngs['target_shank_id'] = _ngs['target_shank_id'] + 1
    _ngs['condition'] = 'pos+NGS'
    combined = pd.concat([_bl, _ngs], ignore_index=True)
    combined['target_shank_id'] = combined['target_shank_id'].astype(int)

    # Prep: SC_x bin labels
    _locs = (
        cell_classifications[cell_classifications['cell_type'] == _ttype]
        [['mouse', 'day', 'cluster_id', 'SC_x']]
        .rename(columns={'cluster_id': 'target_cluster_id'})
        .copy()
    )
    _locs['SC_x'] = pd.to_numeric(_locs['SC_x'], errors='coerce')
    _locs = _locs.dropna(subset=['SC_x'])
    _locs['SC_x_bin'] = pd.cut(_locs['SC_x'].abs(), bins=bin_edges, labels=bin_labels, right=True)
    _locs = _locs.dropna(subset=['SC_x_bin'])
    _locs['SC_x_bin_label'] = pd.Categorical(_locs['SC_x_bin'].astype(str), categories=bin_labels, ordered=True)

    combined_bins = combined.merge(
        _locs[['mouse', 'day', 'target_cluster_id', 'SC_x_bin_label']],
        on=['mouse', 'day', 'target_cluster_id'], how='inner'
    )

    # --- Row 1: by shank ---
    ax = axes[0, col*2]
    sns.boxplot(data=combined, x='target_shank_id', y='pR2_cv',
                hue='condition', hue_order=_hue_order, order=shank_order,
                palette=_palette, width=_box_width_hue, fliersize=0, linecolor='#333333', ax=ax)
    sns.stripplot(data=combined, x='target_shank_id', y='pR2_cv',
                  hue='condition', hue_order=_hue_order, order=shank_order,
                  palette={'pos': '#888888', 'pos+NGS': _col},
                  alpha=0.25, jitter=True, size=2.5, dodge=True, ax=ax, legend=False)
    for patch in ax.patches[:len(shank_order)]:
        patch.set_hatch('///'); patch.set_edgecolor('#333333')
    ax.set_xlabel('Shank', fontsize=9)
    ax.set_ylabel('pR²', fontsize=11)
    ax.set_ylim(-0.05, 0.5)
    ax.set_title('')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if ax.get_legend(): ax.get_legend().remove()

    # Annotate N (sessions) and n (cells) in top right of 1st and 3rd column, 1st row
    if col in [0, 1]:
        N = combined['mouse'].astype(str) + '_' + combined['day'].astype(str)
        N = N.nunique()
        n = combined['target_cluster_id'].nunique()
        ax.text(0.98, 0.98, f'N={N}\nn={n}', ha='right', va='top', color='grey', fontsize=11, transform=ax.transAxes, fontweight='normal', bbox=dict(facecolor='none', edgecolor='none', pad=0))

    # --- Row 1: ΔpR² by shank ---
    ax = axes[0, col*2+1]
    diff_shank = pd.merge(
        _bl, _ngs, on=['mouse', 'day', 'target_cluster_id', 'target_shank_id'],
        suffixes=('_pos', '_ngs')
    )
    diff_shank['delta'] = diff_shank['pR2_cv_ngs'] - diff_shank['pR2_cv_pos']
    sns.boxplot(data=diff_shank, x='target_shank_id', y='delta',
                color=_col, width=_box_width_single, fliersize=0, linecolor='#333333', ax=ax, order=shank_order)
    sns.stripplot(data=diff_shank, x='target_shank_id', y='delta',
                  color=_col, alpha=0.25, jitter=True, size=2.5, ax=ax, order=shank_order)
    for patch in ax.patches[:len(shank_order)]:
        patch.set_hatch('///'); patch.set_edgecolor('#333333')
    ax.set_xlabel('Shank', fontsize=9)
    ax.set_ylabel('ΔpR²', fontsize=11)
    ax.set_ylim(-0.05, 0.5)
    ax.set_title('')
    ax.set_yticklabels([])  # Remove y-axis tick labels
    ax.axhline(0, color='#888', lw=1, ls='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Add compact legend in top-right inside axes, with title "covariate"
    legend_elements = [
        Patch(facecolor='white', edgecolor='#333333', hatch='///', label='pos'),
        Patch(facecolor=_col, edgecolor='#333333', label='pos+NGS'),
        Patch(facecolor=_col, edgecolor='#333333', hatch='///', label='ΔpR²')
    ]
    leg = ax.legend(
        handles=legend_elements,
        loc='upper right',
        bbox_to_anchor=(0.98, 0.98),
        frameon=False,
        fontsize=9,
        title="covariate",
        title_fontsize=10,
        borderaxespad=0.2,
        handlelength=1.2,
        handleheight=1.2,
        borderpad=0.7,
        labelspacing=0.4,
        columnspacing=0.8
    )

    # --- Row 2: by bin ---
    ax = axes[1, col*2]
    sns.boxplot(data=combined_bins, x='SC_x_bin_label', y='pR2_cv',
                hue='condition', hue_order=_hue_order, order=bin_labels,
                palette=_palette, width=_box_width_hue, fliersize=0, linecolor='#333333', ax=ax)
    sns.stripplot(data=combined_bins, x='SC_x_bin_label', y='pR2_cv',
                  hue='condition', hue_order=_hue_order, order=bin_labels,
                  palette={'pos': '#888888', 'pos+NGS': _col},
                  alpha=0.25, jitter=True, size=2.5, dodge=True, ax=ax, legend=False)
    for patch in ax.patches[:len(bin_labels)]:
        patch.set_hatch('///'); patch.set_edgecolor('#333333')
    ax.set_xlabel('M-L bin (µm)', fontsize=9)
    ax.set_ylabel('pR²', fontsize=11)
    ax.set_ylim(-0.05, 0.5)
    ax.set_title('')
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if ax.get_legend(): ax.get_legend().remove()

    # --- Row 2: ΔpR² by bin ---
    ax = axes[1, col*2+1]
    _bl_bins = combined_bins[combined_bins['condition'] == 'pos']
    _ngs_bins = combined_bins[combined_bins['condition'] == 'pos+NGS']
    diff_bin = pd.merge(
        _bl_bins, _ngs_bins,
        on=['mouse', 'day', 'target_cluster_id', 'SC_x_bin_label'],
        suffixes=('_pos', '_ngs')
    )
    diff_bin['delta'] = diff_bin['pR2_cv_ngs'] - diff_bin['pR2_cv_pos']
    sns.boxplot(data=diff_bin, x='SC_x_bin_label', y='delta',
                color=_col, width=_box_width_single, fliersize=0, linecolor='#333333', ax=ax, order=bin_labels)
    sns.stripplot(data=diff_bin, x='SC_x_bin_label', y='delta',
                  color=_col, alpha=0.25, jitter=True, size=2.5, ax=ax, order=bin_labels)
    for patch in ax.patches[:len(bin_labels)]:
        patch.set_hatch('///'); patch.set_edgecolor('#333333')
    ax.set_xlabel('M-L bin (µm)', fontsize=9)
    ax.set_ylabel('ΔpR²', fontsize=11)
    ax.set_ylim(-0.05, 0.5)
    ax.set_title('')
    ax.set_yticklabels([])  # Remove y-axis tick labels
    ax.axhline(0, color='#888', lw=1, ls='--')
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# --- First row, last column: NGS cell pool size by shank (all targets) ---
ngs_count_df = results_df[
    (results_df['covariate_type'] == 'pos+ngs_shank')
].copy()
ngs_count_df['cov_shank_id'] = ngs_count_df['cov_shank_id'].astype(int)

ax = axes[0, 4]
if len(ngs_count_df) == 0:
    ax.axis('off')
    print("No data available — skipping plot.")
else:
    shank_order0 = [0, 1, 2, 3]
    sns.boxplot(data=ngs_count_df, x='cov_shank_id', y='n_ngs_cells',
                order=shank_order0, color='white',
                width=0.5, fliersize=0, linecolor='#333333', ax=ax)
    sns.stripplot(data=ngs_count_df, x='cov_shank_id', y='n_ngs_cells',
                  order=shank_order0, color='grey',
                  alpha=0.2, jitter=True, size=2.5, ax=ax)
    ax.set_xlabel('Shank', fontsize=10)
    ax.set_ylabel('number of NGS cells used in model', fontsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# --- Last column, last row: ΔpR² vs distance to PaS/MEC border plot ---
_par_ids = structure_set[
    structure_set['acronym'].str.match(r'^PAR', na=False)
]['id'].values

_par_mask = np.isin(annotations_set, _par_ids)
_par_voxels = np.argwhere(_par_mask)
if len(_par_voxels) == 0:
    raise RuntimeError("No PAR voxels found — check structure_set acronyms.")
_par_tree = cKDTree(_par_voxels)

_bl = results_df[
    results_df['covariate_type'] == 'pos'
][['mouse', 'day', 'target_cluster_id', 'target_shank_id', 'target_cell_type', 'pR2_cv']].copy()

_ngs_sh = results_df[
    (results_df['covariate_type'] == 'pos+ngs_shank') &
    (results_df['cov_shank_id'] == results_df['target_shank_id'])
][['mouse', 'day', 'target_cluster_id', 'target_shank_id', 'pR2_cv']].copy()

_delta = pd.merge(
    _bl, _ngs_sh,
    on=['mouse', 'day', 'target_cluster_id', 'target_shank_id'],
    suffixes=('_pos', '_ngs')
)
_delta['delta_pR2'] = _delta['pR2_cv_ngs'] - _delta['pR2_cv_pos']

_cc = cell_classifications[['mouse', 'day', 'cluster_id', 'brain_region',
                             'SC_x', 'SC_y', 'SC_z']].copy()
for _col in ['SC_x', 'SC_y', 'SC_z']:
    _cc[_col] = pd.to_numeric(_cc[_col], errors='coerce')

_delta_ent = _delta.merge(
    _cc.rename(columns={'cluster_id': 'target_cluster_id'}),
    on=['mouse', 'day', 'target_cluster_id'],
    how='inner'
)
_delta_ent = _delta_ent[
    _delta_ent['brain_region'].str.contains('ENT', na=False)
].dropna(subset=['SC_x', 'SC_y', 'SC_z']).copy()

_dists_um = []
for _, _row in _delta_ent.iterrows():
    _brain_coord_SC = np.array([_row['SC_z'], _row['SC_y'], -_row['SC_x']])
    _brain_coord_CCF = StereoToCCF(_brain_coord_SC)
    _ccf_pixel = np.round(_brain_coord_CCF / 10).astype(int)
    _dist_px, _ = _par_tree.query(_ccf_pixel)
    _dists_um.append(float(_dist_px) * 10.0)

_delta_ent = _delta_ent.copy()
_delta_ent['dist_to_par_um'] = _dists_um

is_gc = _delta_ent['target_cell_type'] == 'GC'
is_ngs = _delta_ent['target_cell_type'] == 'NG'

ax = axes[1, 4]
plot_df = _delta_ent[is_gc | is_ngs].copy()
plot_df['cell_type'] = np.where(plot_df['target_cell_type'] == 'GC', 'GC', 'NGS')

for cell_type, color in zip(['NGS', 'GC'], [ngs_color, gc_color]):
    sub = plot_df[plot_df['cell_type'] == cell_type]
    ax.scatter(
        sub['dist_to_par_um'], sub['delta_pR2'],
        alpha=0.2, s=2.5, edgecolor=None, linewidths=0.5, color=color, label=f'{cell_type} points'
    )
    if len(sub) > 1:
        x = sub['dist_to_par_um'].values
        y = sub['delta_pR2'].values
        idx = np.argsort(x)
        x_sorted = x[idx]
        y_sorted = y[idx]
        slope, intercept, r, p, stderr = stats.linregress(x, y)
        y_pred = intercept + slope * x_sorted
        n = len(x)
        y_err = stats.sem(y)
        ci = 1.96 * y_err
        ax.plot(x_sorted, y_pred, color=color, lw=2, label=f'{cell_type} fit')
        ax.fill_between(x_sorted, y_pred - ci, y_pred + ci, color=color, alpha=0.18)

ax.axhline(0, color='#aaa', lw=1, ls='--')
ax.set_xlabel('Minimum distance from \n Para-MEC border (µm)')
ax.set_ylabel('ΔpR²')
ax.set_ylim(-0.05, 0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend().remove()
ax.set_yticks([0.4, 0.3, 0.2, 0.1, 0])
ax.set_yticklabels(['0.4', '0.3', '0.2', '0.1', '0'])

# --- Final figure labels ---
fig.text(0.09, 0.72, 'by shank', va='center', ha='center', rotation=90, fontsize=13, fontweight='bold')
fig.text(0.09, 0.32, 'by medial-lateral bin', va='center', ha='center', rotation=90, fontsize=13, fontweight='bold')
fig.text(0.24, 0.97, 'GC target cells', ha='center', va='bottom', fontsize=13, fontweight='bold')
fig.text(0.64, 0.97, 'NGS target cells', ha='center', va='bottom', fontsize=13, fontweight='bold')
plt.savefig('figure5_part2.pdf', dpi=500)
plt.show()

## Example sessions for per-session detail plots

Sessions were ranked by Spearman ρ between absolute shank distance and Δ pR² (computed across all GC × shank rows per session). The five sessions below best represent the global trend of **decreasing Δ pR² with increasing shank distance from the target grid cell**, and are used as illustrative examples in the figures that follow.

| Session | ρ (shank dist vs Δ pR²) | p | N grid cells | Δ pR² same shank | Δ pR² other shanks | diff |
|---|---|---|---|---|---|---|
| **M29 D23** | −0.720 | <0.0001 | 27 | 0.043 | 0.019 | 0.024 |
| **M25 D25** | −0.559 | <0.0001 | 44 | 0.074 | 0.025 | 0.048 |
| **M29 D25** | −0.548 | <0.0001 | 12 | 0.083 | 0.028 | 0.055 |
| **M26 D19** | −0.512 | <0.0001 | 47 | 0.110 | 0.035 | 0.076 |
| **M21 D16** | −0.435 | <0.0001 | 22 | 0.048 | 0.019 | 0.029 |

**Selection criteria:**
- Strongest negative Spearman ρ (indicating steepest distance-dependent drop in predictive power)
- Significant at p < 0.01
- ≥ 10 grid cells to ensure within-session estimates are stable


In [ ]:
import pickle

# ════════════════════════════════════════════════════════════════════════════
# Example trace pre-computation for the first column of the summary figure
# (M25 D25 — first example session)
# ════════════════════════════════════════════════════════════════════════════

# ── User-configurable ─────────────────────────────────────────────────────
# Set to a specific cluster_id int to pin the target cell, or leave as None
# for a reproducible random choice (seed=42).
TRACE_GC_CLUSTER_ID  = None   # e.g. 214
TRACE_NGS_CLUSTER_ID = None   # e.g. 340

# Time window (in bin indices; each bin = time_bs ms)
# Default: 200 bins → 20 s window starting 20 s into the recording
TRACE_WINDOW_BINS = (200, 400)

TRACE_MOUSE, TRACE_DAY = 25, 25

# ── Load session data ─────────────────────────────────────────────────────
import random as _random
_shank_order_ex = [0, 1, 2, 3]

print(f"Loading M{TRACE_MOUSE} D{TRACE_DAY} for example traces...")
_tcs_ex, _tcs_time_ex, _, _, _beh_ex, _cls_ex = compute_vr_tcs(
    TRACE_MOUSE, TRACE_DAY,
    apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path,
)
_last_et_ex = _cls_ex[_cls_ex.index[0]].count(bin_size=time_bs, time_units='ms').index[-1]
_ep_ex      = nap.IntervalSet(start=0, end=_last_et_ex, time_units='s')
_dt_ex      = np.array(
    _beh_ex['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=_ep_ex)
    - ((_beh_ex['trial_number'][0] - 1) * tl)
)
_pos_ex = _dt_ex % tl
if np.any(np.isnan(_pos_ex)):
    _pos_ex = pd.Series(_dt_ex).ffill().bfill().values % tl

# ── Cell tables with shank IDs ────────────────────────────────────────────
_sgcs_ex = cell_classifications[
    (cell_classifications['mouse'] == TRACE_MOUSE) &
    (cell_classifications['day']   == TRACE_DAY) &
    (cell_classifications['cell_type'] == 'GC')
].copy()
_sngs_ex = cell_classifications[
    (cell_classifications['mouse'] == TRACE_MOUSE) &
    (cell_classifications['day']   == TRACE_DAY) &
    (cell_classifications['cell_type'] == 'NG')
].copy()
_sgcs_ex = reconstruct_shank_id(_sgcs_ex, TRACE_MOUSE, colname='probe_x')
_sngs_ex = reconstruct_shank_id(_sngs_ex, TRACE_MOUSE, colname='probe_x')

_ngs_by_shank_ex = {
    sh: grp['cluster_id'].astype(int).values
    for sh, grp in _sngs_ex.groupby('shank_id')
    if any(c in _tcs_time_ex for c in grp['cluster_id'].astype(int))
}

# ── Pick target cells ─────────────────────────────────────────────────────
_valid_gc_ex  = [c for c in _sgcs_ex['cluster_id'].astype(int) if c in _tcs_time_ex]
_valid_ngs_ex = [c for c in _sngs_ex['cluster_id'].astype(int) if c in _tcs_time_ex]
_rng = _random.Random(42)
_trace_gc_id  = TRACE_GC_CLUSTER_ID  if TRACE_GC_CLUSTER_ID  is not None else _rng.choice(_valid_gc_ex)
_trace_ngs_id = TRACE_NGS_CLUSTER_ID if TRACE_NGS_CLUSTER_ID is not None else _rng.choice(_valid_ngs_ex)
print(f"  GC  target : {_trace_gc_id}  (pool: {len(_valid_gc_ex)} cells)")
print(f"  NGS target : {_trace_ngs_id}  (pool: {len(_valid_ngs_ex)} cells)")

# ── Fit models (pos only + pos+NGS shank 0–3) with pickle cache ───────────
_trace_cache = os.path.join(
    '/Users/harryclark/Documents/spatial-manifolds/data',
    f'example_traces_M{TRACE_MOUSE}D{TRACE_DAY}_GC{_trace_gc_id}_NG{_trace_ngs_id}.pkl',
)

def _fit_trace_models_ex(target_id):
    y = np.array(_tcs_time_ex[target_id])
    T = len(y)
    p = _pos_ex[:T]
    if len(p) < T:
        p = np.pad(p, (0, T - len(p)), mode='edge')
    out = {'y': y}

    # position only
    y_hat, pR2_folds = xgb_history.fit_cv(p[:, None], y, verbose=0, continuous_folds=True)
    out['pos'] = (y_hat, float(np.nanmean(pR2_folds)))

    # pos + all NGS cells on each shank
    for sh in _shank_order_ex:
        key = f'pos_sh{sh}'
        if sh not in _ngs_by_shank_ex:
            out[key] = (np.full(T, np.nan), np.nan)
            continue
        valid = [c for c in _ngs_by_shank_ex[sh] if c in _tcs_time_ex and c != target_id]
        if not valid:
            out[key] = (np.full(T, np.nan), np.nan)
            continue
        ngs_stack = np.vstack([
            np.pad(
                np.array(_tcs_time_ex[c])[:T],
                (0, max(0, T - len(np.array(_tcs_time_ex[c])[:T]))),
                mode='constant',
            )
            for c in valid
        ]).T
        x = np.column_stack([p, ngs_stack])
        y_hat, pR2_folds = xgb_history.fit_cv(x, y, verbose=0, continuous_folds=True)
        out[key] = (y_hat, float(np.nanmean(pR2_folds)))
    return out

if os.path.exists(_trace_cache):
    with open(_trace_cache, 'rb') as _f:
        _ct = pickle.load(_f)
    _trace_results_gc  = _ct['gc']
    _trace_results_ngs = _ct['ngs']
    print("Loaded trace models from cache.")
else:
    print("Fitting trace models for GC target  (this may take a few minutes)...")
    _trace_results_gc  = _fit_trace_models_ex(_trace_gc_id)
    print("Fitting trace models for NGS target ...")
    _trace_results_ngs = _fit_trace_models_ex(_trace_ngs_id)
    with open(_trace_cache, 'wb') as _f:
        pickle.dump({'gc': _trace_results_gc, 'ngs': _trace_results_ngs}, _f)
    print("Saved to cache.")

print(f"Window: bins {TRACE_WINDOW_BINS[0]}–{TRACE_WINDOW_BINS[1]} "
      f"({(TRACE_WINDOW_BINS[1] - TRACE_WINDOW_BINS[0]) * time_bs / 1000:.1f} s)")


In [ ]:
import pickle

# ════════════════════════════════════════════════════════════════════════════
# Pre-compute example traces for the v2 figure  (M26 D19)
# ════════════════════════════════════════════════════════════════════════════

# ── User-configurable ─────────────────────────────────────────────────────
TRACE2_GC_CLUSTER_ID  = None   # set to int to pin, e.g. 88
TRACE2_NGS_CLUSTER_ID = None   # set to int to pin, e.g. 210

# 10 time windows spread across the recording (bin indices; 1 bin = time_bs ms)
# Each window is 20 s (2000 bins at 10 ms).  Edit start points to taste.
TRACE2_WINDOWS = [
    (2000,   4000),   # 20 – 40 s
    (5000,   7000),   # 50 – 70 s
    (8000,   10000),  # 80 – 100 s
    (11000,  13000),  # 110 – 130 s
    (14000,  16000),  # 140 – 160 s
    (17000,  19000),  # 170 – 190 s
    (20000,  22000),  # 200 – 220 s
    (23000,  25000),  # 230 – 250 s
    (26000,  28000),  # 260 – 280 s
    (29000,  31000),  # 290 – 310 s
]

TRACE2_MOUSE, TRACE2_DAY = 26, 19

# ── Load session data ─────────────────────────────────────────────────────
import random as _random2
_shank_order_ex2 = [0, 1, 2, 3]

print(f"Loading M{TRACE2_MOUSE} D{TRACE2_DAY} for example traces (v2)...")
_tcs_ex2, _tcs_time_ex2, _, _, _beh_ex2, _cls_ex2 = compute_vr_tcs(
    TRACE2_MOUSE, TRACE2_DAY,
    apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path,
)
_last_et_ex2 = _cls_ex2[_cls_ex2.index[0]].count(bin_size=time_bs, time_units='ms').index[-1]
_ep_ex2 = nap.IntervalSet(start=0, end=_last_et_ex2, time_units='s')
_dt_ex2 = np.array(
    _beh_ex2['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=_ep_ex2)
    - ((_beh_ex2['trial_number'][0] - 1) * tl)
)
_pos_ex2 = _dt_ex2 % tl
if np.any(np.isnan(_pos_ex2)):
    _pos_ex2 = pd.Series(_dt_ex2).ffill().bfill().values % tl

# ── Cell tables with shank IDs ────────────────────────────────────────────
_sgcs_ex2 = cell_classifications[
    (cell_classifications['mouse'] == TRACE2_MOUSE) &
    (cell_classifications['day']   == TRACE2_DAY) &
    (cell_classifications['cell_type'] == 'GC')
].copy()
_sngs_ex2 = cell_classifications[
    (cell_classifications['mouse'] == TRACE2_MOUSE) &
    (cell_classifications['day']   == TRACE2_DAY) &
    (cell_classifications['cell_type'] == 'NG')
].copy()
_sgcs_ex2 = reconstruct_shank_id(_sgcs_ex2, TRACE2_MOUSE, colname='probe_x')
_sngs_ex2 = reconstruct_shank_id(_sngs_ex2, TRACE2_MOUSE, colname='probe_x')

_ngs_by_shank_ex2 = {
    sh: grp['cluster_id'].astype(int).values
    for sh, grp in _sngs_ex2.groupby('shank_id')
    if any(c in _tcs_time_ex2 for c in grp['cluster_id'].astype(int))
}

# ── Pick target cells ─────────────────────────────────────────────────────
_valid_gc_ex2  = [c for c in _sgcs_ex2['cluster_id'].astype(int) if c in _tcs_time_ex2]
# NGS example restricted to shank 2 (3rd shank), same as the GC example
_valid_ngs_ex2 = [c for c in _sngs_ex2[_sngs_ex2['shank_id'] == 2]['cluster_id'].astype(int) if c in _tcs_time_ex2]
print(f"  Available NGS cells on shank 2: {_valid_ngs_ex2}")
_rng2_gc  = _random2.Random(42)
_rng2_ngs = _random2.Random(7)
_trace2_gc_id  = TRACE2_GC_CLUSTER_ID  if TRACE2_GC_CLUSTER_ID  is not None else _rng2_gc.choice(_valid_gc_ex2)
_trace2_ngs_id = TRACE2_NGS_CLUSTER_ID if TRACE2_NGS_CLUSTER_ID is not None else _rng2_ngs.choice(_valid_ngs_ex2)
print(f"  GC  target : {_trace2_gc_id}  (pool: {len(_valid_gc_ex2)} cells)")
print(f"  NGS target : {_trace2_ngs_id}  (pool: {len(_valid_ngs_ex2)} cells, shank 2)")

# ── Fit models (pos only + pos+NGS shank 0–3) with pickle cache ───────────
_trace2_cache = os.path.join(
    '/Users/harryclark/Documents/spatial-manifolds/data',
    f'example_traces_v2_M{TRACE2_MOUSE}D{TRACE2_DAY}_GC{_trace2_gc_id}_NG{_trace2_ngs_id}.pkl',
)

def _fit_trace_models_ex2(target_id):
    y = np.array(_tcs_time_ex2[target_id])
    T = len(y)
    p = _pos_ex2[:T]
    if len(p) < T:
        p = np.pad(p, (0, T - len(p)), mode='edge')
    out = {'y': y}

    # position only
    y_hat, pR2_folds = xgb_history.fit_cv(p[:, None], y, verbose=0, continuous_folds=True)
    out['pos'] = (y_hat, float(np.nanmean(pR2_folds)))

    # pos + all NGS cells on each shank
    for sh in _shank_order_ex2:
        key = f'pos_sh{sh}'
        if sh not in _ngs_by_shank_ex2:
            out[key] = (np.full(T, np.nan), np.nan)
            continue
        valid = [c for c in _ngs_by_shank_ex2[sh] if c in _tcs_time_ex2 and c != target_id]
        if not valid:
            out[key] = (np.full(T, np.nan), np.nan)
            continue
        ngs_stack = np.vstack([
            np.pad(
                np.array(_tcs_time_ex2[c])[:T],
                (0, max(0, T - len(np.array(_tcs_time_ex2[c])[:T]))),
                mode='constant',
            )
            for c in valid
        ]).T
        x = np.column_stack([p, ngs_stack])
        y_hat, pR2_folds = xgb_history.fit_cv(x, y, verbose=0, continuous_folds=True)
        out[key] = (y_hat, float(np.nanmean(pR2_folds)))
    return out

if os.path.exists(_trace2_cache):
    with open(_trace2_cache, 'rb') as _f:
        _ct2 = pickle.load(_f)
    _trace2_results_gc  = _ct2['gc']
    _trace2_results_ngs = _ct2['ngs']
    print("Loaded trace models from cache.")
else:
    print("Fitting trace models for GC target  (this may take a few minutes)...")
    _trace2_results_gc  = _fit_trace_models_ex2(_trace2_gc_id)
    print("Fitting trace models for NGS target ...")
    _trace2_results_ngs = _fit_trace_models_ex2(_trace2_ngs_id)
    with open(_trace2_cache, 'wb') as _f:
        pickle.dump({'gc': _trace2_results_gc, 'ngs': _trace2_results_ngs}, _f)
    print("Saved to cache.")

print(f"  Windows : {len(TRACE2_WINDOWS)} x "
      f"{(TRACE2_WINDOWS[0][1] - TRACE2_WINDOWS[0][0]) * time_bs / 1000:.0f} s each")


In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import spikeinterface.full as si
from probeinterface.plotting import plot_probegroup

# ════════════════════════════════════════════════════════════════════════════
# Summary figure v2 — M26 D19 only, with multi-window trace column
# Each window in TRACE2_WINDOWS produces one complete figure.
# ════════════════════════════════════════════════════════════════════════════

# ── User-configurable ────────────────────────────────────────────────────────
_pred_scale_gc   = 15          # scale factor for GC prediction traces
_pred_scale_ngs  = 5           # scale factor for NGS prediction traces

# ── Config ─────────────────────────────────────────────────────────────────
_v2_sessions = [
    {'mouse': 26, 'day': 19},
]
_v2_shank_order  = [0, 1, 2, 3]
_v2_shank_labels = [1, 2, 3, 4]

# ── Guard ──────────────────────────────────────────────────────────────────
if '_trace2_results_gc' not in dir() or _trace2_results_gc is None:
    print("WARNING: run the v2 pre-compute cell first.")
    _trace2_results_gc = _trace2_results_ngs = None

# ── Load probe group ───────────────────────────────────────────────────────
_has_probe2   = False
_probe_group2 = None
try:
    _pp2  = "/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Chris/Cohort12/derivatives/"
    _epd2 = [f.path for f in os.scandir(f"{_pp2}25/") if f.is_dir()][25]
    _sa2  = si.load_sorting_analyzer(f"{_epd2}/full/kilosort4/kilosort4_sa", load_extensions=False)
    _probe_group2 = _sa2.get_probegroup()
    _has_probe2 = True
except Exception as _e2:
    print(f"Probe group unavailable ({_e2}); using schematic.")

# ── Δ pR² data ─────────────────────────────────────────────────────────────
_bl_v2 = (
    results_df[results_df['covariate_type'] == 'pos']
    [['mouse', 'day', 'target_cluster_id', 'pR2_cv']]
    .rename(columns={'pR2_cv': 'pR2_baseline'})
)
_cv_v2 = results_df[results_df['covariate_type'] == 'pos+ngs_shank'].copy()
_cv_v2 = _cv_v2.merge(_bl_v2, on=['mouse', 'day', 'target_cluster_id'], how='left')
_cv_v2['delta_pR2']       = _cv_v2['pR2_cv'] - _cv_v2['pR2_baseline']
_cv_v2 = _cv_v2.dropna(subset=['cov_shank_id', 'delta_pR2'])
_cv_v2['cov_shank_id']    = _cv_v2['cov_shank_id'].astype(int)
_cv_v2['target_shank_id'] = _cv_v2['target_shank_id'].astype(int)

_type_cfg_v2 = {
    'GC': dict(cmap=white_to_hex_cmap(gc_color),  tgt_label='GC',  xlabel='Target GC shank'),
    'NG': dict(cmap=white_to_hex_cmap(ngs_color),      tgt_label='NGS', xlabel='Target NGS shank'),
}

def _mat_norm_v2(vals):
    vmax = max(float(np.nanmax(vals[~np.isnan(vals)])), 0.01) if not np.all(np.isnan(vals)) else 0.01
    return mcolors.Normalize(vmin=0, vmax=vmax)

def _build_pop_v2(cv_sub):
    _cm  = (cv_sub.groupby(['mouse', 'day', 'cov_shank_id', 'target_shank_id'])['delta_pR2']
            .mean().reset_index())
    _ids = _cm[['mouse', 'day']].drop_duplicates().values.tolist()
    _mats = []
    for _m, _d in _ids:
        _s  = _cm[(_cm['mouse'] == _m) & (_cm['day'] == _d)]
        _sm = (_s.groupby(['cov_shank_id', 'target_shank_id'])['delta_pR2']
               .mean().unstack(fill_value=np.nan))
        _sm = _sm.reindex(index=_v2_shank_order, columns=_v2_shank_order)
        _mats.append(_sm.values.astype(float))
    if not _mats:
        _e = np.full((4, 4), np.nan)
        return _e, _e, 0
    _stk  = np.stack(_mats, axis=0)
    _mean = np.nanmean(_stk, axis=0)
    _sem  = np.nanstd(_stk, axis=0) / np.sqrt(np.sum(~np.isnan(_stk), axis=0))
    return _mean, _sem, len(_ids)

_pop_v2 = {}
for _tt in ['GC', 'NG']:
    _sub = _cv_v2[_cv_v2['target_cell_type'] == _tt] if 'target_cell_type' in _cv_v2.columns else _cv_v2
    _m, _s, _n = _build_pop_v2(_sub)
    _pop_v2[_tt] = dict(mean=_m, sem=_s, n=_n)
    print(f"v2 Population ({_tt}): {_n} sessions")

# ── Pre-build annotation cache ─────────────────────────────────────────────
_ann_cache_v2 = {}
for _si2 in _v2_sessions:
    _mid2 = _si2['mouse']
    if _mid2 not in _ann_cache_v2:
        try:
            _a2, _ac2, _ax2, _ay2 = make_annotations(
                _mid2, path='/Users/harryclark/Documents/spatial-manifolds/data')
            _ann_cache_v2[_mid2] = (_ac2, _ax2, _ay2)
        except Exception:
            _ann_cache_v2[_mid2] = None

# ── Pre-build cell tables ──────────────────────────────────────────────────
_sess_cells_v2 = {}
for _si2 in _v2_sessions:
    _mid2, _did2 = _si2['mouse'], _si2['day']
    _sg = cell_classifications[
        (cell_classifications['mouse'] == _mid2) & (cell_classifications['day'] == _did2) &
        (cell_classifications['cell_type'] == 'GC')].copy()
    _sn = cell_classifications[
        (cell_classifications['mouse'] == _mid2) & (cell_classifications['day'] == _did2) &
        (cell_classifications['cell_type'] == 'NG')].copy()
    _sg = reconstruct_shank_id(_sg, _mid2, colname='probe_x')
    _sn = reconstruct_shank_id(_sn, _mid2, colname='probe_x')
    _sess_cells_v2[(_mid2, _did2)] = (_sg, _sn)

# ── Helper: draw probe map ─────────────────────────────────────────────────
def _draw_probe_v2(axp, mid2, did2):
    sgcs2, sngs2 = _sess_cells_v2[(mid2, did2)]

    if _ann_cache_v2.get(mid2) is not None:
        _acd2, _axd2, _ayd2 = _ann_cache_v2[mid2]
        for _col2 in np.unique(_acd2):
            _bp2 = extract_border(_acd2, _col2, only_border=False)
            axp.scatter(_axd2[_bp2[:, 1]], _ayd2[_bp2[:, 0]],
                        color=_col2, s=1, rasterized=True, zorder=0)

    if _has_probe2 and _probe_group2 is not None:
        plot_probegroup(_probe_group2, ax=axp,
                        probe_shape_kwargs={'alpha': 0.35, 'color': 'lightgrey', 'edgecolor': '#555555'},
                        contacts_kargs={'alpha': 0.0})
    else:
        for _sx2 in [0, 250, 500, 750]:
            axp.fill_betweenx([0, 3840], _sx2 - 30, _sx2 + 30,
                              color='lightgrey', edgecolor='#555555',
                              alpha=0.35, linewidth=0.6, zorder=1)

    axp.scatter(sngs2['probe_x'], sngs2['probe_y'], c=ngs_color, s=18, zorder=3, alpha=0.85, linewidths=0)
    axp.scatter(sgcs2['probe_x'], sgcs2['probe_y'], c=gc_color,  s=18, zorder=4, alpha=0.85, linewidths=0)

    # Star markers for the trace target cells (trace session only)
    if mid2 == TRACE2_MOUSE and did2 == TRACE2_DAY:
        for _star_id, _star_color in [(_trace2_gc_id, gc_color), (_trace2_ngs_id, ngs_color)]:
            _star_row = cell_classifications[
                (cell_classifications['mouse'] == mid2) &
                (cell_classifications['day']   == did2) &
                (cell_classifications['cluster_id'] == _star_id)
            ]
            if len(_star_row) > 0:
                axp.scatter(
                    float(_star_row['probe_x'].values[0]),
                    float(_star_row['probe_y'].values[0]),
                    marker='*', s=140, c=_star_color,
                    zorder=6, linewidths=0.6, edgecolors='white',
                )

    _all_y2 = np.concatenate([sgcs2['probe_y'].dropna().values, sngs2['probe_y'].dropna().values])
    if len(_all_y2) > 0:
        axp.set_ylim(max(0, _all_y2.min() - 200), _all_y2.max() + 200)
    axp.set_xlim(-200, 1000)
    remove_all_from_ax(axp)
    axp.set_title(f'M{mid2} D{did2}', fontsize=8, fontweight='bold', pad=3)

    _yb2 = axp.get_ylim()[0]
    _lo2 = (axp.get_ylim()[1] - axp.get_ylim()[0]) * 0.04
    _dsx = {0: 0, 1: 250, 2: 500, 3: 750}
    for _sh2 in _v2_shank_order:
        _sn2 = sngs2[sngs2['shank_id'] == _sh2] if 'shank_id' in sngs2.columns else pd.DataFrame()
        _sg2 = sgcs2[sgcs2['shank_id'] == _sh2] if 'shank_id' in sgcs2.columns else pd.DataFrame()
        _nn2, _ng2 = len(_sn2), len(_sg2)
        _xr2 = (float(_sn2['probe_x'].median()) if _nn2 > 0
                else float(_sg2['probe_x'].median()) if _ng2 > 0
                else _dsx.get(_sh2, _sh2 * 250))
        _pf2 = 'n=' if _sh2 == 0 else ''
        axp.text(_xr2, _yb2,        f'{_pf2}{_nn2}', ha='center', va='top', fontsize=7, color=ngs_color, clip_on=False)
        axp.text(_xr2, _yb2 + _lo2, f'{_pf2}{_ng2}', ha='center', va='top', fontsize=7, color=gc_color,  clip_on=False)


# ── Helper: legend panel in last col, row 0 ───────────────────────────────
def _draw_legend_panel(ax):
    ax.axis('off')
    ax.legend(
        handles=[
            Line2D([0],[0], marker='o', color='w', markerfacecolor=ngs_color,     markersize=7,  label='NGS'),
            Line2D([0],[0], marker='o', color='w', markerfacecolor=gc_color,      markersize=7,  label='GC'),
            Line2D([0],[0], marker='*', color='w', markerfacecolor=gc_color,  markersize=10, label=f'GC target ({_trace2_gc_id})'),
            Line2D([0],[0], marker='*', color='w', markerfacecolor=ngs_color,     markersize=10, label=f'NGS target ({_trace2_ngs_id})'),
        ],
        loc='center', fontsize=7, frameon=False,
        handletextpad=0.4, labelspacing=0.5,
    )

def _draw_raster_v2(
    ax, window_bins, cls_grp, sgcs, sngs, shank_order, time_bs_ms,
    gc_col, ngs_col, target_gc_id=None, target_ngs_id=None,
    shank_divider_gap=5, gc_ngs_gap=10
):
    
    _s_bin, _e_bin = window_bins
    _center_s = (_s_bin + _e_bin) / 2.0 * (time_bs_ms / 1000.0)
    _half_dur = 10
    _t0 = max(0.0, _center_s - _half_dur)
    _t1 = _center_s + _half_dur

    # Restrict spikes to window
    _ep_r = nap.IntervalSet(start=_t0, end=_t1, time_units='s')
    _cls_r = cls_grp.restrict(_ep_r)
    _spk_dict = {}
    for _uid in _cls_r.index:
        try:
            _spk_dict[int(_uid)] = _cls_r[_uid].times()
        except Exception:
            _spk_dict[int(_uid)] = np.array([])

    _avail = set(_spk_dict.keys())

    # Per-shank cell lists
    _gc_by_sh, _ng_by_sh = {}, {}
    for _sh in shank_order:
        _gc_by_sh[_sh] = [
            c for c in (sgcs[sgcs['shank_id'] == _sh]
                        .sort_values('probe_y', ascending=False)['cluster_id']
                        .astype(int).tolist() if 'shank_id' in sgcs.columns else [])
            if c in _avail
        ]
        _ng_by_sh[_sh] = [
            c for c in (sngs[sngs['shank_id'] == _sh]
                        .sort_values('probe_y', ascending=False)['cluster_id']
                        .astype(int).tolist() if 'shank_id' in sngs.columns else [])
            if c in _avail
        ]

    any_gc = any(_gc_by_sh[_sh] for _sh in shank_order)
    any_ngs = any(_ng_by_sh[_sh] for _sh in shank_order)

    # Build row assignments: GCs, gap, NGSs
    _rows = []
    for _shi, _sh in enumerate(shank_order):
        for _cid in _gc_by_sh[_sh]:
            _rows.append((_cid, gc_col, _cid == target_gc_id))
        if _shi < len(shank_order) - 1 and _gc_by_sh[_sh]:
            for _ in range(shank_divider_gap):
                _rows.append((None, None, False))
    if any_gc and any_ngs:
        for _ in range(gc_ngs_gap):
            _rows.append((None, None, False))
    for _shi, _sh in enumerate(shank_order):
        for _cid in _ng_by_sh[_sh]:
            _rows.append((_cid, ngs_col, _cid == target_ngs_id))
        if _shi < len(shank_order) - 1 and _ng_by_sh[_sh]:
            for _ in range(shank_divider_gap):
                _rows.append((None, None, False))

    _n_rows = len(_rows)
    if _n_rows == 0:
        ax.text(0.5, 0.5, 'No cells', ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='grey')
        ax.axis('off')
        return

    # Plot spikes
    _dur = _t1 - _t0
    for _ri, (_cid, _col, _is_tgt) in enumerate(_rows):
        if _cid is None:
            continue
        _spk = _spk_dict.get(_cid, np.array([]))
        if len(_spk) == 0:
            continue
        _t_rel = _spk - _t0
        _ms = 1.5 if _is_tgt else 0.4
        _alp = 1.0 if _is_tgt else 0.65
        ax.scatter(_t_rel, np.full(len(_t_rel), _ri),
                   c=_col, s=_ms, linewidths=0, alpha=_alp, rasterized=True)

    # Shank labels (centered)
    _row_cursor = 0
    for _shi, _sh in enumerate(shank_order):
        _n = len(_gc_by_sh[_sh])
        if _n > 0:
            ax.text(_dur * 1.01, _row_cursor + _n / 2.0 - 0.5,
                    f'Sh{_sh + 1}', ha='left', va='center',
                    fontsize=6, color=gc_col, clip_on=False)
        _row_cursor += _n
        if _shi < len(shank_order) - 1 and _gc_by_sh[_sh]:
            _row_cursor += shank_divider_gap
    if any_gc and any_ngs:
        _row_cursor += gc_ngs_gap
    for _shi, _sh in enumerate(shank_order):
        _n = len(_ng_by_sh[_sh])
        if _n > 0:
            ax.text(_dur * 1.01, _row_cursor + _n / 2.0 - 0.5,
                    f'Sh{_sh + 1}', ha='left', va='center',
                    fontsize=6, color=ngs_col, clip_on=False)
        _row_cursor += _n
        if _shi < len(shank_order) - 1 and _ng_by_sh[_sh]:
            _row_cursor += shank_divider_gap

    # Group labels (rotated)
    _gc_count = sum(len(_gc_by_sh[_sh]) for _sh in shank_order)
    _ngs_count = sum(len(_ng_by_sh[_sh]) for _sh in shank_order)
    _gc_gaps = shank_divider_gap * sum(1 for _shi, _sh in enumerate(shank_order)
                         if _shi < len(shank_order) - 1 and bool(_gc_by_sh[_sh]))
    _ngs_gaps = shank_divider_gap * sum(1 for _shi, _sh in enumerate(shank_order)
                         if _shi < len(shank_order) - 1 and bool(_ng_by_sh[_sh]))
    _gc_block = _gc_count + _gc_gaps
    _ngs_block = _ngs_count + _ngs_gaps
    if _gc_count > 0:
        ax.text(-_dur * 0.03, _gc_block / 2.0 - 0.5, 'GC',
                ha='right', va='center', fontsize=6.5, color=gc_col,
                fontweight='bold', clip_on=False, rotation=90)
    if _ngs_count > 0:
        _ngs_start = _gc_block + (gc_ngs_gap if any_gc and any_ngs else 0)
        ax.text(-_dur * 0.03, _ngs_start + _ngs_block / 2.0 - 0.5, 'NGS',
                ha='right', va='center', fontsize=6.5, color=ngs_col,
                fontweight='bold', clip_on=False, rotation=90)

    # Axis limits and scale bar
    ax.set_xlim(0, _dur)
    ax.set_ylim(_n_rows - 0.5, -0.5)
    _sb_y = _n_rows + _n_rows * 0.05
    ax.plot([0, 5.0], [_sb_y, _sb_y], color='black', lw=1.5, clip_on=False)
    ax.text(2.5, _sb_y + _n_rows * 0.02, '5 s',
            ha='center', va='top', fontsize=6, clip_on=False)
    ax.set_title('Spike raster  (20 seconds)', fontsize=7, fontweight='bold', pad=3)
    ax.axis('off')

# ── Helper: draw single-window trace subplot ───────────────────────────────
def _draw_single_window(ax, trace_results, ttype, window_bins, pred_scale=10, show_labels=True):
    if trace_results is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='grey')
        ax.axis('off')
        return

    s, e    = window_bins
    y_full  = trace_results['y']
    T_win   = e - s
    t_sec   = np.arange(T_win) * (time_bs / 1000.0)

    _trace_color = gc_color if ttype == 'GC' else ngs_color

    keys   = ['y', 'pos'] + [f'pos_sh{sh}' for sh in [0, 1, 2, 3]]
    colors = ['#aaaaaa', _trace_color] + [_trace_color] * 4
    labels = ['True', 'Pos'] + [f'Pos+Sh{sh + 1}' for sh in [0, 1, 2, 3]]
    lws    = [1.2, 1.1] + [1.0] * 4
    n      = len(keys)

    # pR² values
    pr2_vals = {}
    for key in keys:
        if key == 'y':
            continue
        raw = trace_results.get(key)
        pr2_vals[key] = raw[1] if raw is not None else np.nan

    # offset scale from this window's true firing rate
    y_win = y_full[s:e] if len(y_full) >= e else np.full(T_win, np.nan)
    offset_step  = max(np.nanmax(np.abs(y_win)) * 1.7, 1.0) if not np.all(np.isnan(y_win)) else 1.0
    y_win_smooth = gaussian_filter_nan(y_win, sigma=3)

    for i, (key, color, label, lw) in enumerate(zip(keys, colors, labels, lws)):
        offset = (n - 1 - i) * offset_step
        if key == 'y':
            ax.plot(t_sec, y_win + offset, color=color, lw=lw, alpha=0.7)
            ax.plot(t_sec, y_win_smooth + offset, color=_trace_color, lw=1.8, alpha=0.9)
        else:
            raw = trace_results.get(key)
            if raw is None:
                continue
            pred_full = raw[0]
            trace = pred_full[s:e] if len(pred_full) >= e else np.full(T_win, np.nan)
            ax.plot(t_sec, trace * pred_scale + offset, color=color, lw=lw, alpha=0.88)

        if show_labels:
            if key == 'y':
                ax.text(-t_sec[-1] * 0.03, offset + offset_step * 0.12, 'Smoothed',
                        color=_trace_color, va='center', ha='right', fontsize=5.5, clip_on=False)
                ax.text(-t_sec[-1] * 0.03, offset - offset_step * 0.12, label,
                        color=color, va='center', ha='right', fontsize=5.5, clip_on=False)
            else:
                ax.text(-t_sec[-1] * 0.03, offset, label,
                        color=color, va='center', ha='right', fontsize=5.5, clip_on=False)
            if key != 'y':
                pr2v = pr2_vals.get(key, np.nan)
                if pr2v is not None and np.isfinite(float(pr2v)):
                    ax.text(t_sec[-1] + t_sec[-1] * 0.03, offset,
                            f'{pr2v:.2f}', color=color, va='center', ha='left',
                            fontsize=5.5, clip_on=False)

    # 5-second scale bar
    _bar_y = -offset_step * 0.55
    ax.plot([0, 5.0], [_bar_y, _bar_y], color='black', lw=1.5, clip_on=False)
    ax.text(2.5, _bar_y - offset_step * 0.12, '5 s', ha='center', va='top', fontsize=6)

    uid2 = _trace2_gc_id if ttype == 'GC' else _trace2_ngs_id
    start_s = s * time_bs / 1000
    ax.set_title(f'Unit {uid2} ({ttype}) — {start_s:.0f} s  ×{pred_scale}',
                 fontsize=7, fontweight='bold', pad=3)
    ax.axis('off')

# ── Helper: draw Δ pR² matrix row ─────────────────────────────────────────
def _draw_mat_row_v2(fig_ref, axes_list, ax_pop, ttype, show_pop_title=True):
    cfg     = _type_cfg_v2[ttype]
    _cmap   = cfg['cmap']
    _sub_cv = _cv_v2[_cv_v2['target_cell_type'] == ttype] if 'target_cell_type' in _cv_v2.columns else _cv_v2

    for ci2, sess_info2 in enumerate(_v2_sessions):
        mid2, did2 = sess_info2['mouse'], sess_info2['day']
        ax_mat = axes_list[ci2]
        sess_cv = _sub_cv[(_sub_cv['mouse'] == mid2) & (_sub_cv['day'] == did2)].copy()
        mat = (sess_cv.groupby(['cov_shank_id', 'target_shank_id'])['delta_pR2']
               .mean().unstack(fill_value=np.nan))
        mat = mat.reindex(index=_v2_shank_order, columns=_v2_shank_order)
        mv  = mat.values.astype(float)
        _nm = _mat_norm_v2(mv)
        im  = ax_mat.imshow(mv, cmap=_cmap, norm=_nm, aspect='equal', interpolation='nearest')
        fig_ref.colorbar(im, ax=ax_mat, fraction=0.046, pad=0.04, shrink=0.8).ax.tick_params(labelsize=5.5)
        for r in range(4):
            for c in range(4):
                v = mv[r, c]
                if not np.isnan(v):
                    tc = 'white' if _nm(max(v, 0)) > 0.6 else '#222222'
                    ax_mat.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=7, color=tc)
        ax_mat.set_xticks(range(4)); ax_mat.set_xticklabels(_v2_shank_labels, fontsize=6.5)
        ax_mat.tick_params(axis='x', length=0)
        ax_mat.set_yticks(range(4))
        ax_mat.set_yticklabels(_v2_shank_labels if ci2 == 0 else [], fontsize=6.5)
        ax_mat.tick_params(axis='y', length=0)
        for sp in ax_mat.spines.values():
            sp.set_visible(False)
        ax_mat.set_xlabel(cfg['xlabel'], fontsize=6.5)
        if ci2 == 0:
            ax_mat.set_ylabel('Cov NGS shank', fontsize=6.5)

    _pm  = _pop_v2[ttype]['mean']
    _ps  = _pop_v2[ttype]['sem']
    _pn  = _pop_v2[ttype]['n']
    _pnm = _mat_norm_v2(_pm)
    im_p = ax_pop.imshow(_pm, cmap=_cmap, norm=_pnm, aspect='equal', interpolation='nearest')
    fig_ref.colorbar(im_p, ax=ax_pop, fraction=0.046, pad=0.04, shrink=0.8).ax.tick_params(labelsize=5.5)
    for r in range(4):
        for c in range(4):
            v = _pm[r, c]; sv = _ps[r, c]
            if not np.isnan(v):
                tc = 'white' if _pnm(max(v, 0)) > 0.6 else '#222222'
                ax_pop.text(c, r, f'{v:.2f}\n±{sv:.2f}',
                            ha='center', va='center', fontsize=6.5, color=tc, linespacing=1.4)
    ax_pop.set_xticks(range(4)); ax_pop.set_xticklabels(_v2_shank_labels, fontsize=6.5)
    ax_pop.tick_params(axis='x', length=0)
    ax_pop.set_yticks(range(4)); ax_pop.set_yticklabels(_v2_shank_labels, fontsize=6.5)
    ax_pop.tick_params(axis='y', length=0)
    for sp in ax_pop.spines.values():
        sp.set_visible(False)
    ax_pop.set_xlabel(cfg['xlabel'], fontsize=6.5)
    ax_pop.set_ylabel('Cov NGS shank', fontsize=6.5)
    if show_pop_title:
        ax_pop.set_title(f'Population\nmean ± SEM\n(N={_pn})', fontsize=8, fontweight='bold', pad=3)

# ── Loop over windows — one complete figure per window ────────────────────
_col_w   = 2.8
_trace_w = _col_w * 3   # trace column is 2× as wide
_probe_h = 3.2
_mat_h   = 3.2          # fixed height per matrix row (single window)
n_cols_v2 = 3           # trace + 1 session + population

os.makedirs(fig_path, exist_ok=True)

for _wi, _win in enumerate(TRACE2_WINDOWS):
    _s_bin, _e_bin = _win
    _start_s = _s_bin * time_bs / 1000

    _fv2 = plt.figure(figsize=(_trace_w + _col_w * 2, _probe_h + _mat_h * 2))
    _gs_v2 = gridspec.GridSpec(
        3, n_cols_v2, figure=_fv2,
        height_ratios=[_probe_h, _mat_h, _mat_h],
        width_ratios=[_trace_w, _col_w, _col_w],
        hspace=0.20, wspace=0.32,
    )

    # Row 0: raster (col 0) + probe map (col 1) + legend (col 2)
    _ax_raster = _fv2.add_subplot(_gs_v2[0, 0])
    _draw_raster_v2(
        _ax_raster, _win,
        _cls_ex2, _sgcs_ex2, _sngs_ex2,
        _shank_order_ex2, time_bs,
        gc_color, ngs_color,
        target_gc_id=_trace2_gc_id, target_ngs_id=_trace2_ngs_id,
    )
    for _ci2, _si2 in enumerate(_v2_sessions):
        _draw_probe_v2(
            _fv2.add_subplot(_gs_v2[0, _ci2 + 1]),
            _si2['mouse'], _si2['day'],
        )
    _ax_legend = _fv2.add_subplot(_gs_v2[0, n_cols_v2 - 1])
    _draw_legend_panel(_ax_legend)

    # Row 1: GC trace + GC matrices + GC population
    _ax_gc_tr  = _fv2.add_subplot(_gs_v2[1, 0])
    _ax_gc_ms  = [_fv2.add_subplot(_gs_v2[1, _ci2 + 1]) for _ci2 in range(len(_v2_sessions))]
    _ax_gc_p2  = _fv2.add_subplot(_gs_v2[1, n_cols_v2 - 1])

    # Row 2: NGS trace + NGS matrices + NGS population
    _ax_ngs_tr = _fv2.add_subplot(_gs_v2[2, 0])
    _ax_ngs_ms = [_fv2.add_subplot(_gs_v2[2, _ci2 + 1]) for _ci2 in range(len(_v2_sessions))]
    _ax_ngs_p2 = _fv2.add_subplot(_gs_v2[2, n_cols_v2 - 1])

    _draw_single_window(_ax_gc_tr,  _trace2_results_gc,  'GC', _win, pred_scale=_pred_scale_gc)
    _draw_mat_row_v2(_fv2, _ax_gc_ms,  _ax_gc_p2,  'GC')

    _draw_single_window(_ax_ngs_tr, _trace2_results_ngs, 'NG', _win, pred_scale=_pred_scale_ngs)
    _draw_mat_row_v2(_fv2, _ax_ngs_ms, _ax_ngs_p2, 'NG', show_pop_title=False)

    _fname = f'ngs_shank_summary_v2_win{_wi + 1:02d}_{int(_start_s):04d}s.pdf'
    _fv2.savefig(fig_path + _fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved {_fname}")


In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import spikeinterface.full as si
from probeinterface.plotting import plot_probegroup

# ════════════════════════════════════════════════════════════════════════════
# Summary figure v2 — M26 D19 only, with multi-window trace column
# Each window in TRACE2_WINDOWS produces one complete figure.
# ════════════════════════════════════════════════════════════════════════════

# ── User-configurable ────────────────────────────────────────────────────────
_pred_scale_gc   = 15          # scale factor for GC prediction traces
_pred_scale_ngs  = 5           # scale factor for NGS prediction traces

# ── Config ─────────────────────────────────────────────────────────────────
_v2_sessions = [
    {'mouse': 26, 'day': 19},
]
_v2_shank_order  = [0, 1, 2, 3]
_v2_shank_labels = [1, 2, 3, 4]

# ── Guard ──────────────────────────────────────────────────────────────────
if '_trace2_results_gc' not in dir() or _trace2_results_gc is None:
    print("WARNING: run the v2 pre-compute cell first.")
    _trace2_results_gc = _trace2_results_ngs = None

# ── Load probe group ───────────────────────────────────────────────────────
_has_probe2   = False
_probe_group2 = None
try:
    _pp2  = "/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Chris/Cohort12/derivatives/"
    _epd2 = [f.path for f in os.scandir(f"{_pp2}25/") if f.is_dir()][25]
    _sa2  = si.load_sorting_analyzer(f"{_epd2}/full/kilosort4/kilosort4_sa", load_extensions=False)
    _probe_group2 = _sa2.get_probegroup()
    _has_probe2 = True
except Exception as _e2:
    print(f"Probe group unavailable ({_e2}); using schematic.")

# ── Δ pR² data ─────────────────────────────────────────────────────────────
_bl_v2 = (
    results_df[results_df['covariate_type'] == 'pos']
    [['mouse', 'day', 'target_cluster_id', 'pR2_cv']]
    .rename(columns={'pR2_cv': 'pR2_baseline'})
)
_cv_v2 = results_df[results_df['covariate_type'] == 'pos+ngs_shank'].copy()
_cv_v2 = _cv_v2.merge(_bl_v2, on=['mouse', 'day', 'target_cluster_id'], how='left')
_cv_v2['delta_pR2']       = _cv_v2['pR2_cv'] - _cv_v2['pR2_baseline']
_cv_v2 = _cv_v2.dropna(subset=['cov_shank_id', 'delta_pR2'])
_cv_v2['cov_shank_id']    = _cv_v2['cov_shank_id'].astype(int)
_cv_v2['target_shank_id'] = _cv_v2['target_shank_id'].astype(int)

_type_cfg_v2 = {
    'GC': dict(cmap=white_to_hex_cmap(gc_color),  tgt_label='GC',  xlabel='Target GC shank'),
    'NG': dict(cmap=white_to_hex_cmap(ngs_color),      tgt_label='NGS', xlabel='Target NGS shank'),
}

def _mat_norm_v2(vals):
    vmax = max(float(np.nanmax(vals[~np.isnan(vals)])), 0.01) if not np.all(np.isnan(vals)) else 0.01
    return mcolors.Normalize(vmin=0, vmax=vmax)

def _build_pop_v2(cv_sub):
    _cm  = (cv_sub.groupby(['mouse', 'day', 'cov_shank_id', 'target_shank_id'])['delta_pR2']
            .mean().reset_index())
    _ids = _cm[['mouse', 'day']].drop_duplicates().values.tolist()
    _mats = []
    for _m, _d in _ids:
        _s  = _cm[(_cm['mouse'] == _m) & (_cm['day'] == _d)]
        _sm = (_s.groupby(['cov_shank_id', 'target_shank_id'])['delta_pR2']
               .mean().unstack(fill_value=np.nan))
        _sm = _sm.reindex(index=_v2_shank_order, columns=_v2_shank_order)
        _mats.append(_sm.values.astype(float))
    if not _mats:
        _e = np.full((4, 4), np.nan)
        return _e, _e, 0
    _stk  = np.stack(_mats, axis=0)
    _mean = np.nanmean(_stk, axis=0)
    _sem  = np.nanstd(_stk, axis=0) / np.sqrt(np.sum(~np.isnan(_stk), axis=0))
    return _mean, _sem, len(_ids)

_pop_v2 = {}
for _tt in ['GC', 'NG']:
    _sub = _cv_v2[_cv_v2['target_cell_type'] == _tt] if 'target_cell_type' in _cv_v2.columns else _cv_v2
    _m, _s, _n = _build_pop_v2(_sub)
    _pop_v2[_tt] = dict(mean=_m, sem=_s, n=_n)
    print(f"v2 Population ({_tt}): {_n} sessions")

# ── Pre-build annotation cache ─────────────────────────────────────────────
_ann_cache_v2 = {}
for _si2 in _v2_sessions:
    _mid2 = _si2['mouse']
    if _mid2 not in _ann_cache_v2:
        try:
            _a2, _ac2, _ax2, _ay2 = make_annotations(
                _mid2, path='/Users/harryclark/Documents/spatial-manifolds/data')
            _ann_cache_v2[_mid2] = (_ac2, _ax2, _ay2)
        except Exception:
            _ann_cache_v2[_mid2] = None

# ── Pre-build cell tables ──────────────────────────────────────────────────
_sess_cells_v2 = {}
for _si2 in _v2_sessions:
    _mid2, _did2 = _si2['mouse'], _si2['day']
    _sg = cell_classifications[
        (cell_classifications['mouse'] == _mid2) & (cell_classifications['day'] == _did2) &
        (cell_classifications['cell_type'] == 'GC')].copy()
    _sn = cell_classifications[
        (cell_classifications['mouse'] == _mid2) & (cell_classifications['day'] == _did2) &
        (cell_classifications['cell_type'] == 'NG')].copy()
    _sg = reconstruct_shank_id(_sg, _mid2, colname='probe_x')
    _sn = reconstruct_shank_id(_sn, _mid2, colname='probe_x')
    _sess_cells_v2[(_mid2, _did2)] = (_sg, _sn)

# ── Helper: draw probe map ─────────────────────────────────────────────────
def _draw_probe_v2(axp, mid2, did2):
    sgcs2, sngs2 = _sess_cells_v2[(mid2, did2)]

    if _ann_cache_v2.get(mid2) is not None:
        _acd2, _axd2, _ayd2 = _ann_cache_v2[mid2]
        for _col2 in np.unique(_acd2):
            _bp2 = extract_border(_acd2, _col2, only_border=False)
            axp.scatter(_axd2[_bp2[:, 1]], _ayd2[_bp2[:, 0]],
                        color=_col2, s=1, rasterized=True, zorder=0)

    if _has_probe2 and _probe_group2 is not None:
        plot_probegroup(_probe_group2, ax=axp,
                        probe_shape_kwargs={'alpha': 0.35, 'color': 'lightgrey', 'edgecolor': '#555555'},
                        contacts_kargs={'alpha': 0.0})
    else:
        for _sx2 in [0, 250, 500, 750]:
            axp.fill_betweenx([0, 3840], _sx2 - 30, _sx2 + 30,
                              color='lightgrey', edgecolor='#555555',
                              alpha=0.35, linewidth=0.6, zorder=1)

    axp.scatter(sngs2['probe_x'], sngs2['probe_y'], c=ngs_color, s=18, zorder=3, alpha=0.85, linewidths=0)
    axp.scatter(sgcs2['probe_x'], sgcs2['probe_y'], c=gc_color,  s=18, zorder=4, alpha=0.85, linewidths=0)

    # Star markers for the trace target cells (trace session only)
    if mid2 == TRACE2_MOUSE and did2 == TRACE2_DAY:
        for _star_id, _star_color in [(_trace2_gc_id, gc_color), (_trace2_ngs_id, ngs_color)]:
            _star_row = cell_classifications[
                (cell_classifications['mouse'] == mid2) &
                (cell_classifications['day']   == did2) &
                (cell_classifications['cluster_id'] == _star_id)
            ]
            if len(_star_row) > 0:
                axp.scatter(
                    float(_star_row['probe_x'].values[0]),
                    float(_star_row['probe_y'].values[0]),
                    marker='*', s=140, c=_star_color,
                    zorder=6, linewidths=0.6, edgecolors='white',
                )

    _all_y2 = np.concatenate([sgcs2['probe_y'].dropna().values, sngs2['probe_y'].dropna().values])
    if len(_all_y2) > 0:
        axp.set_ylim(max(0, _all_y2.min() - 200), _all_y2.max() + 200)
    axp.set_xlim(-200, 1000)
    remove_all_from_ax(axp)
    axp.set_title(f'M{mid2} D{did2}', fontsize=8, fontweight='bold', pad=3)

    _yb2 = axp.get_ylim()[0]
    _lo2 = (axp.get_ylim()[1] - axp.get_ylim()[0]) * 0.04
    _dsx = {0: 0, 1: 250, 2: 500, 3: 750}
    for _sh2 in _v2_shank_order:
        _sn2 = sngs2[sngs2['shank_id'] == _sh2] if 'shank_id' in sngs2.columns else pd.DataFrame()
        _sg2 = sgcs2[sgcs2['shank_id'] == _sh2] if 'shank_id' in sgcs2.columns else pd.DataFrame()
        _nn2, _ng2 = len(_sn2), len(_sg2)
        _xr2 = (float(_sn2['probe_x'].median()) if _nn2 > 0
                else float(_sg2['probe_x'].median()) if _ng2 > 0
                else _dsx.get(_sh2, _sh2 * 250))
        _pf2 = 'n=' if _sh2 == 0 else ''
        axp.text(_xr2, _yb2,        f'{_pf2}{_nn2}', ha='center', va='top', fontsize=7, color=ngs_color, clip_on=False)
        axp.text(_xr2, _yb2 + _lo2, f'{_pf2}{_ng2}', ha='center', va='top', fontsize=7, color=gc_color,  clip_on=False)


# ── Helper: legend panel in last col, row 0 ───────────────────────────────
def _draw_legend_panel(ax):
    ax.axis('off')
    ax.legend(
        handles=[
            Line2D([0],[0], marker='o', color='w', markerfacecolor=ngs_color,     markersize=7,  label='NGS'),
            Line2D([0],[0], marker='o', color='w', markerfacecolor=gc_color,      markersize=7,  label='GC'),
            Line2D([0],[0], marker='*', color='w', markerfacecolor=gc_color,  markersize=10, label=f'GC target ({_trace2_gc_id})'),
            Line2D([0],[0], marker='*', color='w', markerfacecolor=ngs_color,     markersize=10, label=f'NGS target ({_trace2_ngs_id})'),
        ],
        loc='center', fontsize=7, frameon=False,
        handletextpad=0.4, labelspacing=0.5,
    )


# ── Helper: spike raster with integrated position trace ───────────────────
def _draw_raster_v2(
    ax, window_bins, cls_grp, sgcs, sngs, shank_order, time_bs_ms,
    gc_col, ngs_col, target_gc_id=None, target_ngs_id=None,
    shank_divider_gap=5, gc_ngs_gap=10,
    pos_binned=None, pos_color='tab:blue', pos_height_rows=10,
):
    """
    Draw a spike raster with an optional position trace overlaid in the same
    subplot, above the spike rows.

    Parameters
    ----------
    pos_binned : array-like, optional
        Full-session position array binned at time_bs_ms resolution (e.g.
        _pos_ex2).  The function slices the relevant window internally.
    pos_color : str
        Colour for the position trace line.
    pos_height_rows : int
        Number of blank raster rows allocated above the spikes for the
        position trace band.
    """
    _s_bin, _e_bin = window_bins
    _center_s = (_s_bin + _e_bin) / 2.0 * (time_bs_ms / 1000.0)
    _half_dur = 10
    _t0 = max(0.0, _center_s - _half_dur)
    _t1 = _center_s + _half_dur

    # Restrict spikes to window
    _ep_r = nap.IntervalSet(start=_t0, end=_t1, time_units='s')
    _cls_r = cls_grp.restrict(_ep_r)
    _spk_dict = {}
    for _uid in _cls_r.index:
        try:
            _spk_dict[int(_uid)] = _cls_r[_uid].times()
        except Exception:
            _spk_dict[int(_uid)] = np.array([])

    _avail = set(_spk_dict.keys())

    # Per-shank cell lists
    _gc_by_sh, _ng_by_sh = {}, {}
    for _sh in shank_order:
        _gc_by_sh[_sh] = [
            c for c in (sgcs[sgcs['shank_id'] == _sh]
                        .sort_values('probe_y', ascending=False)['cluster_id']
                        .astype(int).tolist() if 'shank_id' in sgcs.columns else [])
            if c in _avail
        ]
        _ng_by_sh[_sh] = [
            c for c in (sngs[sngs['shank_id'] == _sh]
                        .sort_values('probe_y', ascending=False)['cluster_id']
                        .astype(int).tolist() if 'shank_id' in sngs.columns else [])
            if c in _avail
        ]

    any_gc  = any(_gc_by_sh[_sh] for _sh in shank_order)
    any_ngs = any(_ng_by_sh[_sh] for _sh in shank_order)

    # Build row assignments: GCs, gap, NGSs
    _rows = []
    for _shi, _sh in enumerate(shank_order):
        for _cid in _gc_by_sh[_sh]:
            _rows.append((_cid, gc_col, _cid == target_gc_id))
        if _shi < len(shank_order) - 1 and _gc_by_sh[_sh]:
            for _ in range(shank_divider_gap):
                _rows.append((None, None, False))
    if any_gc and any_ngs:
        for _ in range(gc_ngs_gap):
            _rows.append((None, None, False))
    for _shi, _sh in enumerate(shank_order):
        for _cid in _ng_by_sh[_sh]:
            _rows.append((_cid, ngs_col, _cid == target_ngs_id))
        if _shi < len(shank_order) - 1 and _ng_by_sh[_sh]:
            for _ in range(shank_divider_gap):
                _rows.append((None, None, False))

    _n_rows = len(_rows)
    _dur    = _t1 - _t0

    if _n_rows == 0:
        ax.text(0.5, 0.5, 'No cells', ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='grey')
        ax.axis('off')
        return

    # ── Position trace overlay ────────────────────────────────────────────
    # Occupies rows [-pos_height_rows, 0) above the spike rows.
    # The y-axis is inverted (row 0 at top of spike area), so negative rows
    # appear above the spikes.
    _pos_height = pos_height_rows if pos_binned is not None else 0
    if pos_binned is not None:
        _b0 = int(round(_t0 * 1000.0 / time_bs_ms))
        _b1 = int(round(_t1 * 1000.0 / time_bs_ms))
        _pv = np.array(pos_binned[_b0:_b1], dtype=float)
        _pt = np.arange(len(_pv)) * (time_bs_ms / 1000.0)
        _pmin, _pmax = np.nanmin(_pv), np.nanmax(_pv)
        _prange = _pmax - _pmin if _pmax > _pmin else 1.0
        # Map to y in [-_pos_height + 0.5, -0.5]: higher pos → higher on plot
        _py = -0.5 - (_pv - _pmin) / _prange * (_pos_height - 1.0)
        ax.plot(_pt, _py, color=pos_color, lw=1.0, zorder=10, alpha=0.85)
        # Rotated "Pos" label on the left, centred in the position band
        ax.text(-_dur * 0.03, -_pos_height / 2.0, 'Pos',
                ha='right', va='center', fontsize=6.5, color=pos_color,
                fontweight='bold', clip_on=False, rotation=90)
        # Thin separator line between position trace and spike rows
        ax.axhline(0, color='#cccccc', lw=0.5, zorder=1)

    # ── Plot spikes ───────────────────────────────────────────────────────
    for _ri, (_cid, _col, _is_tgt) in enumerate(_rows):
        if _cid is None:
            continue
        _spk = _spk_dict.get(_cid, np.array([]))
        if len(_spk) == 0:
            continue
        _t_rel = _spk - _t0
        _ms  = 1.5 if _is_tgt else 0.4
        _alp = 1.0 if _is_tgt else 0.65
        ax.scatter(_t_rel, np.full(len(_t_rel), _ri),
                   c=_col, s=_ms, linewidths=0, alpha=_alp, rasterized=True)

    # ── Shank labels (centred, right side) ───────────────────────────────
    _row_cursor = 0
    for _shi, _sh in enumerate(shank_order):
        _n = len(_gc_by_sh[_sh])
        if _n > 0:
            ax.text(_dur * 1.01, _row_cursor + _n / 2.0 - 0.5,
                    f'Sh{_sh + 1}', ha='left', va='center',
                    fontsize=6, color=gc_col, clip_on=False)
        _row_cursor += _n
        if _shi < len(shank_order) - 1 and _gc_by_sh[_sh]:
            _row_cursor += shank_divider_gap
    if any_gc and any_ngs:
        _row_cursor += gc_ngs_gap
    for _shi, _sh in enumerate(shank_order):
        _n = len(_ng_by_sh[_sh])
        if _n > 0:
            ax.text(_dur * 1.01, _row_cursor + _n / 2.0 - 0.5,
                    f'Sh{_sh + 1}', ha='left', va='center',
                    fontsize=6, color=ngs_col, clip_on=False)
        _row_cursor += _n
        if _shi < len(shank_order) - 1 and _ng_by_sh[_sh]:
            _row_cursor += shank_divider_gap

    # ── Group labels (rotated, left side) ────────────────────────────────
    _gc_count  = sum(len(_gc_by_sh[_sh]) for _sh in shank_order)
    _ngs_count = sum(len(_ng_by_sh[_sh]) for _sh in shank_order)
    _gc_gaps   = shank_divider_gap * sum(1 for _shi, _sh in enumerate(shank_order)
                         if _shi < len(shank_order) - 1 and bool(_gc_by_sh[_sh]))
    _ngs_gaps  = shank_divider_gap * sum(1 for _shi, _sh in enumerate(shank_order)
                         if _shi < len(shank_order) - 1 and bool(_ng_by_sh[_sh]))
    _gc_block  = _gc_count + _gc_gaps
    _ngs_block = _ngs_count + _ngs_gaps
    if _gc_count > 0:
        ax.text(-_dur * 0.03, _gc_block / 2.0 - 0.5, 'GC',
                ha='right', va='center', fontsize=6.5, color=gc_col,
                fontweight='bold', clip_on=False, rotation=90)
    if _ngs_count > 0:
        _ngs_start = _gc_block + (gc_ngs_gap if any_gc and any_ngs else 0)
        ax.text(-_dur * 0.03, _ngs_start + _ngs_block / 2.0 - 0.5, 'NGS',
                ha='right', va='center', fontsize=6.5, color=ngs_col,
                fontweight='bold', clip_on=False, rotation=90)

    # ── Axis limits and scale bar ─────────────────────────────────────────
    ax.set_xlim(0, _dur)
    ax.set_ylim(_n_rows - 0.5, -_pos_height - 0.5)   # includes position band
    _sb_y = _n_rows + _n_rows * 0.05
    ax.plot([0, 5.0], [_sb_y, _sb_y], color='black', lw=1.5, clip_on=False)
    ax.text(2.5, _sb_y + _n_rows * 0.02, '5 s',
            ha='center', va='top', fontsize=6, clip_on=False)
    ax.set_title('Spike raster  (20 seconds)', fontsize=7, fontweight='bold', pad=3)
    ax.axis('off')


# ── Helper: draw single-window trace subplot ───────────────────────────────
def _draw_single_window(ax, trace_results, ttype, window_bins, pred_scale=10, show_labels=True):
    if trace_results is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='grey')
        ax.axis('off')
        return

    s, e    = window_bins
    y_full  = trace_results['y']
    T_win   = e - s
    t_sec   = np.arange(T_win) * (time_bs / 1000.0)

    _trace_color = gc_color if ttype == 'GC' else ngs_color

    keys   = ['y', 'pos'] + [f'pos_sh{sh}' for sh in [0, 1, 2, 3]]
    colors = ['#aaaaaa', _trace_color] + [_trace_color] * 4
    labels = ['True', 'Pos'] + [f'Pos+Sh{sh + 1}' for sh in [0, 1, 2, 3]]
    lws    = [1.2, 1.1] + [1.0] * 4
    n      = len(keys)

    # pR² values
    pr2_vals = {}
    for key in keys:
        if key == 'y':
            continue
        raw = trace_results.get(key)
        pr2_vals[key] = raw[1] if raw is not None else np.nan

    # offset scale from this window's true firing rate
    y_win = y_full[s:e] if len(y_full) >= e else np.full(T_win, np.nan)
    offset_step  = max(np.nanmax(np.abs(y_win)) * 1.7, 1.0) if not np.all(np.isnan(y_win)) else 1.0
    y_win_smooth = gaussian_filter_nan(y_win, sigma=3)

    for i, (key, color, label, lw) in enumerate(zip(keys, colors, labels, lws)):
        offset = (n - 1 - i) * offset_step
        if key == 'y':
            ax.plot(t_sec, y_win + offset, color=color, lw=lw, alpha=0.7)
            ax.plot(t_sec, y_win_smooth + offset, color=_trace_color, lw=1.8, alpha=0.9)
        else:
            raw = trace_results.get(key)
            if raw is None:
                continue
            pred_full = raw[0]
            trace = pred_full[s:e] if len(pred_full) >= e else np.full(T_win, np.nan)
            ax.plot(t_sec, trace * pred_scale + offset, color=color, lw=lw, alpha=0.88)

        if show_labels:
            if key == 'y':
                ax.text(-t_sec[-1] * 0.03, offset + offset_step * 0.12, 'Smoothed',
                        color=_trace_color, va='center', ha='right', fontsize=5.5, clip_on=False)
                ax.text(-t_sec[-1] * 0.03, offset - offset_step * 0.12, label,
                        color=color, va='center', ha='right', fontsize=5.5, clip_on=False)
            else:
                ax.text(-t_sec[-1] * 0.03, offset, label,
                        color=color, va='center', ha='right', fontsize=5.5, clip_on=False)
            if key != 'y':
                pr2v = pr2_vals.get(key, np.nan)
                if pr2v is not None and np.isfinite(float(pr2v)):
                    ax.text(t_sec[-1] + t_sec[-1] * 0.03, offset,
                            f'{pr2v:.2f}', color=color, va='center', ha='left',
                            fontsize=5.5, clip_on=False)

    # 5-second scale bar
    _bar_y = -offset_step * 0.55
    ax.plot([0, 5.0], [_bar_y, _bar_y], color='black', lw=1.5, clip_on=False)
    ax.text(2.5, _bar_y - offset_step * 0.12, '5 s', ha='center', va='top', fontsize=6)

    uid2 = _trace2_gc_id if ttype == 'GC' else _trace2_ngs_id
    start_s = s * time_bs / 1000
    ax.set_title(f'Unit {uid2} ({ttype}) — {start_s:.0f} s  ×{pred_scale}',
                 fontsize=7, fontweight='bold', pad=3)
    ax.axis('off')


# ── Helper: draw Δ pR² matrix row ─────────────────────────────────────────
def _draw_mat_row_v2(fig_ref, axes_list, ax_pop, ttype, show_pop_title=True):
    cfg     = _type_cfg_v2[ttype]
    _cmap   = cfg['cmap']
    _sub_cv = _cv_v2[_cv_v2['target_cell_type'] == ttype] if 'target_cell_type' in _cv_v2.columns else _cv_v2

    for ci2, sess_info2 in enumerate(_v2_sessions):
        mid2, did2 = sess_info2['mouse'], sess_info2['day']
        ax_mat = axes_list[ci2]
        sess_cv = _sub_cv[(_sub_cv['mouse'] == mid2) & (_sub_cv['day'] == did2)].copy()
        mat = (sess_cv.groupby(['cov_shank_id', 'target_shank_id'])['delta_pR2']
               .mean().unstack(fill_value=np.nan))
        mat = mat.reindex(index=_v2_shank_order, columns=_v2_shank_order)
        mv  = mat.values.astype(float)
        _nm = _mat_norm_v2(mv)
        im  = ax_mat.imshow(mv, cmap=_cmap, norm=_nm, aspect='equal', interpolation='nearest')
        cb = fig_ref.colorbar(im, ax=ax_mat, fraction=0.046, pad=0.04, shrink=0.8)
        cb.ax.tick_params(labelsize=5.5)
        cb.set_label('ΔpR²', fontsize=6.5)
        for r in range(4):
            for c in range(4):
                v = mv[r, c]
                if not np.isnan(v):
                    tc = 'white' if _nm(max(v, 0)) > 0.6 else '#222222'
                    ax_mat.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=7, color=tc)
        ax_mat.set_xticks(range(4)); ax_mat.set_xticklabels(_v2_shank_labels, fontsize=6.5)
        ax_mat.tick_params(axis='x', length=0)
        ax_mat.set_yticks(range(4))
        ax_mat.set_yticklabels(_v2_shank_labels if ci2 == 0 else [], fontsize=6.5)
        ax_mat.tick_params(axis='y', length=0)
        for sp in ax_mat.spines.values():
            sp.set_visible(False)
        ax_mat.set_xlabel(cfg['xlabel'], fontsize=6.5)
        if ci2 == 0:
            ax_mat.set_ylabel('Cov NGS shank', fontsize=6.5)

    _pm  = _pop_v2[ttype]['mean']
    _ps  = _pop_v2[ttype]['sem']
    _pn  = _pop_v2[ttype]['n']
    _pnm = _mat_norm_v2(_pm)
    im_p = ax_pop.imshow(_pm, cmap=_cmap, norm=_pnm, aspect='equal', interpolation='nearest')
    cb = fig_ref.colorbar(im_p, ax=ax_pop, fraction=0.046, pad=0.04, shrink=0.8)
    cb.ax.tick_params(labelsize=5.5)
    cb.set_label('Mean ΔpR²', fontsize=6.5)
    for r in range(4):
        for c in range(4):
            v = _pm[r, c]; sv = _ps[r, c]
            if not np.isnan(v):
                tc = 'white' if _pnm(max(v, 0)) > 0.6 else '#222222'
                ax_pop.text(c, r, f'{v:.2f}\n±{sv:.2f}',
                            ha='center', va='center', fontsize=6.5, color=tc, linespacing=1.4)
    ax_pop.set_xticks(range(4)); ax_pop.set_xticklabels(_v2_shank_labels, fontsize=6.5)
    ax_pop.tick_params(axis='x', length=0)
    ax_pop.set_yticks(range(4)); ax_pop.set_yticklabels(_v2_shank_labels, fontsize=6.5)
    ax_pop.tick_params(axis='y', length=0)
    for sp in ax_pop.spines.values():
        sp.set_visible(False)
    ax_pop.set_xlabel(cfg['xlabel'], fontsize=6.5)
    ax_pop.set_ylabel('Cov NGS shank', fontsize=6.5)
    if show_pop_title:
        ax_pop.set_title(f'Population\nmean ± SEM\n(N={_pn})', fontsize=8, fontweight='bold', pad=3)


# ── Loop over windows — one complete figure per window ────────────────────
_col_w   = 2.8
_trace_w = _col_w * 3   # trace column is 3× as wide
_probe_h = 3.2
_mat_h   = 3.2          # fixed height per matrix row (single window)
n_cols_v2 = 3           # trace + 1 session + population

os.makedirs(fig_path, exist_ok=True)

for _wi, _win in enumerate(TRACE2_WINDOWS):
    _s_bin, _e_bin = _win
    _start_s = _s_bin * time_bs / 1000

    _fv2 = plt.figure(figsize=(_trace_w + _col_w * 2, _probe_h + _mat_h * 2))
    _gs_v2 = gridspec.GridSpec(
        3, n_cols_v2, figure=_fv2,
        height_ratios=[_probe_h, _mat_h, _mat_h],
        width_ratios=[_trace_w, _col_w, _col_w],
        hspace=0.20, wspace=0.32,
    )

    # Row 0: raster (col 0) + probe map (col 1) + legend (col 2)
    # _pos_ex2 is passed so the position trace is overlaid above the spikes.
    _ax_raster = _fv2.add_subplot(_gs_v2[0, 0])
    _draw_raster_v2(
        _ax_raster, _win,
        _cls_ex2, _sgcs_ex2, _sngs_ex2,
        _shank_order_ex2, time_bs,
        gc_color, ngs_color,
        target_gc_id=_trace2_gc_id, target_ngs_id=_trace2_ngs_id,
        pos_binned=_pos_ex2,          # position overlay
        pos_color='black',
        pos_height_rows=10,
    )
    for _ci2, _si2 in enumerate(_v2_sessions):
        _draw_probe_v2(
            _fv2.add_subplot(_gs_v2[0, _ci2 + 1]),
            _si2['mouse'], _si2['day'],
        )
    _ax_legend = _fv2.add_subplot(_gs_v2[0, n_cols_v2 - 1])
    _draw_legend_panel(_ax_legend)

    # Row 1: GC trace + GC matrices + GC population
    _ax_gc_tr  = _fv2.add_subplot(_gs_v2[1, 0])
    _ax_gc_ms  = [_fv2.add_subplot(_gs_v2[1, _ci2 + 1]) for _ci2 in range(len(_v2_sessions))]
    _ax_gc_p2  = _fv2.add_subplot(_gs_v2[1, n_cols_v2 - 1])

    # Row 2: NGS trace + NGS matrices + NGS population
    _ax_ngs_tr = _fv2.add_subplot(_gs_v2[2, 0])
    _ax_ngs_ms = [_fv2.add_subplot(_gs_v2[2, _ci2 + 1]) for _ci2 in range(len(_v2_sessions))]
    _ax_ngs_p2 = _fv2.add_subplot(_gs_v2[2, n_cols_v2 - 1])

    _draw_single_window(_ax_gc_tr,  _trace2_results_gc,  'GC', _win, pred_scale=_pred_scale_gc)
    _draw_mat_row_v2(_fv2, _ax_gc_ms,  _ax_gc_p2,  'GC')

    _draw_single_window(_ax_ngs_tr, _trace2_results_ngs, 'NG', _win, pred_scale=_pred_scale_ngs)
    _draw_mat_row_v2(_fv2, _ax_ngs_ms, _ax_ngs_p2, 'NG', show_pop_title=False)

    _fname = f'ngs_shank_summary_v2_win{_wi + 1:02d}_{int(_start_s):04d}s.pdf'
    _fv2.savefig(fig_path + _fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved {_fname}")

## Extended figures